In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ra-gnarok/sample_submission.csv
/kaggle/input/competitions/ra-gnarok/Test Data CSV - Sheet1.csv
/kaggle/input/competitions/ra-gnarok/sample_submission.json
/kaggle/input/competitions/ra-gnarok/EU AI Act.pdf


In [2]:
# ==============================
# Module 1.1: Install local dependencies
# 模块 1.1：安装本地依赖
# ==============================

# PyMuPDF is used for local PDF parsing.
# PyMuPDF 用于本地解析 PDF，不需要任何 API。
!pip install -qU pymupdf

# LangChain core document object and text splitter.
# LangChain 的 Document 对象和文本切分器。
!pip install -qU langchain-core langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 9.2 MB/s eta 0:00:00


In [3]:
# ==============================
# Module 1.2: Imports and configuration
# 模块 1.2：导入库与基础配置
# ==============================

import os
import re
import json
from typing import List, Dict, Tuple, Optional
from collections import Counter

import fitz  # PyMuPDF, local PDF parser / 本地 PDF 解析器

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ------------------------------
# Kaggle input directory
# Kaggle 输入目录
# ------------------------------
KAGGLE_INPUT_DIR = "/kaggle/input"


# ------------------------------
# Output path
# 输出路径
# ------------------------------
CHUNK_OUTPUT_PATH = "chunks_legal_structure.jsonl"


# ------------------------------
# Chunking configuration
# 切分配置
# ------------------------------

# Maximum character length for secondary chunks.
# 二级切分后每个 chunk 的最大字符数。
CHUNK_SIZE = 1600

# Overlap keeps context between adjacent chunks.
# overlap 用于保留相邻 chunk 之间的上下文。
CHUNK_OVERLAP = 250


print("✅ Module 1 configuration loaded.")

✅ Module 1 configuration loaded.


In [4]:
# ==============================
# Module 1.3: Locate EU AI Act PDF
# 模块 1.3：自动定位 EU AI Act PDF
# ==============================

def find_pdf_file(root_dir: str = KAGGLE_INPUT_DIR) -> str:
    """
    Find the EU AI Act PDF file under Kaggle input directory.
    在 Kaggle input 目录下查找 EU AI Act PDF 文件。

    Returns:
        str: PDF file path
        返回 PDF 文件路径
    """
    pdf_candidates = []

    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(".pdf"):
                pdf_candidates.append(os.path.join(root, file))

    if not pdf_candidates:
        raise FileNotFoundError("No PDF file found under /kaggle/input.")

    # Prefer file names containing "AI" and "Act".
    # 优先选择文件名中包含 AI 和 Act 的 PDF。
    for path in pdf_candidates:
        filename = os.path.basename(path).lower()
        if "ai" in filename and "act" in filename:
            return path

    # Fallback: return the first PDF file.
    # 兜底：返回第一个 PDF 文件。
    return pdf_candidates[0]


pdf_path = find_pdf_file()
print(f"✅ PDF found: {pdf_path}")

✅ PDF found: /kaggle/input/competitions/ra-gnarok/EU AI Act.pdf


In [5]:
# ==============================
# Module 1.4: Extract and clean PDF text locally
# 模块 1.4：本地解析并清洗 PDF 文本
# ==============================

def clean_page_text(text: str) -> str:
    """
    Clean raw page text extracted from PDF.
    清洗从 PDF 页面提取出来的原始文本。

    We avoid over-cleaning because legal text relies on numbers,
    punctuation, paragraph structure, and article references.
    不要过度清洗法律文本，因为数字、标点、段落结构和法条引用都很重要。
    """
    if not text:
        return ""

    # Remove control characters and strange symbols.
    # 移除控制字符和异常符号。
    text = text.replace("\x0c", "\n")
    text = text.replace("\ufeff", "")
    text = text.replace("￾", "")

    # Normalize spaces.
    # 规范化空格。
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize excessive blank lines.
    # 合并过多空行。
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Strip every line while preserving line breaks.
    # 去掉每行首尾空格，同时保留换行结构。
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)

    return text.strip()


def extract_pdf_pages(pdf_path: str) -> List[Dict]:
    """
    Extract text from PDF page by page using PyMuPDF.
    使用 PyMuPDF 按页提取 PDF 文本。

    Returns:
        List[Dict]: Each item contains page number and text.
        每个元素包含页码和该页文本。
    """
    doc = fitz.open(pdf_path)
    pages = []

    for page_index, page in enumerate(doc):
        # sort=True usually improves reading order.
        # sort=True 通常可以改善文本阅读顺序。
        raw_text = page.get_text("text", sort=True)
        cleaned_text = clean_page_text(raw_text)

        pages.append({
            "page": page_index + 1,
            "text": cleaned_text
        })

    return pages


def build_full_text_with_page_markers(pages: List[Dict]) -> str:
    """
    Combine all pages and insert explicit page markers.
    合并所有页面，并插入显式页码标记。

    Page markers help trace evidence back to the PDF.
    页码标记方便之后追踪证据来自哪一页。
    """
    parts = []

    for item in pages:
        page_no = item["page"]
        page_text = item["text"]

        if page_text:
            parts.append(f"\n\n[PAGE {page_no}]\n{page_text}")

    return "\n".join(parts)


pages = extract_pdf_pages(pdf_path)
full_text = build_full_text_with_page_markers(pages)

print(f"✅ Total pages extracted: {len(pages)}")
print(f"✅ Total characters extracted: {len(full_text)}")

print("\nPreview / 文本预览:")
print(full_text[:1200])

✅ Total pages extracted: 459
✅ Total characters extracted: 599365

Preview / 文本预览:


[PAGE 1]
European Parliament
2019-2024

TEXTS ADOPTED

P9_TA(2024)0138
Artificial Intelligence Act
European Parliament legislative resolution of 13 March 2024 on the proposal for a
regulation of the European Parliament and of the Council on laying down harmonised
rules on Artificial Intelligence (Artificial Intelligence Act) and amending certain Union
Legislative Acts (COM(2021)0206 – C9-0146/2021 – 2021/0106(COD))
(Ordinary legislative procedure: first reading)

The European Parliament,

– having regard to the Commission proposal to Parliament and the Council
(COM(2021)0206),

– having regard to Article 294(2) and Articles 16 and 114 of the Treaty on the
Functioning of the European Union, pursuant to which the Commission submitted the
proposal to Parliament (C9-0146/2021),

– having regard to Article 294(3) of the Treaty on the Functioning of the European Union,

– having regard to the opinion of the 

In [6]:
# ==============================
# Module 1.5: Legal structure detection helpers
# 模块 1.5：法律结构识别辅助函数
# ==============================

# Article header pattern, e.g. "Article 3".
# Article 标题模式，例如 Article 3。
ARTICLE_HEADER_PATTERN = re.compile(
    r"(?im)^\s*Article\s+(\d+[A-Za-z]?)\s*$"
)

# Annex header pattern, e.g. "ANNEX III".
# Annex 标题模式，例如 ANNEX III。
ANNEX_HEADER_PATTERN = re.compile(
    r"(?im)^\s*ANNEX\s+([IVXLC]+)\s*$"
)

# Recital pattern, e.g. "(1) The purpose of this Regulation..."
# Recital 模式，例如以 (1) 开头的立法说明。
RECITAL_PATTERN = re.compile(
    r"(?m)^\((\d+)\)\s+"
)

# Page marker pattern inserted by us.
# 我们自己插入的页码标记。
PAGE_MARKER_PATTERN = re.compile(
    r"\[PAGE\s+(\d+)\]"
)


def infer_pages(text: str) -> Tuple[Optional[int], Optional[int]]:
    """
    Infer page range from page markers.
    从页码标记推断 chunk 的页码范围。
    """
    page_matches = PAGE_MARKER_PATTERN.findall(text)

    if not page_matches:
        return None, None

    page_numbers = [int(p) for p in page_matches]
    return min(page_numbers), max(page_numbers)


def infer_article_number(text: str) -> Optional[str]:
    """
    Infer Article number from chunk text.
    从 chunk 文本中推断 Article 编号。
    """
    match = ARTICLE_HEADER_PATTERN.search(text)

    if match:
        return match.group(1)

    return None


def infer_article_title(text: str) -> Optional[str]:
    """
    Infer Article title.
    推断 Article 标题。

    In many EU legal documents, the title appears right after "Article X".
    在很多欧盟法律文本中，Article 标题通常在 Article X 的下一行。
    """
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    for idx, line in enumerate(lines):
        if re.match(r"(?i)^Article\s+\d+[A-Za-z]?$", line):
            if idx + 1 < len(lines):
                title = lines[idx + 1].strip()

                # Avoid treating page markers as titles.
                # 避免把页码标记误判为标题。
                if not title.startswith("[PAGE"):
                    return title[:180]

            return line[:180]

    return None


def infer_annex_number(text: str) -> Optional[str]:
    """
    Infer Annex number from chunk text.
    从 chunk 文本中推断 Annex 编号。
    """
    match = ANNEX_HEADER_PATTERN.search(text)

    if match:
        return match.group(1)

    return None


def split_text_by_header_positions(text: str, pattern: re.Pattern) -> List[str]:
    """
    Split text according to header positions while preserving headers.
    根据标题位置切分文本，并保留标题本身。

    This is better than simple re.split because the header remains in the chunk.
    这比简单 re.split 更适合法律文本，因为标题会留在 chunk 内。
    """
    matches = list(pattern.finditer(text))

    if not matches:
        return [text]

    sections = []

    # Keep text before the first header.
    # 保留第一个标题之前的文本。
    if matches[0].start() > 0:
        pre_text = text[:matches[0].start()].strip()

        if pre_text:
            sections.append(pre_text)

    # Split from each header to the next header.
    # 从每个标题切到下一个标题。
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        section = text[start:end].strip()

        if section:
            sections.append(section)

    return sections

In [7]:
# ==============================
# Module 1.6: Create LangChain Documents with metadata
# 模块 1.6：创建带法律元数据的 LangChain Document
# ==============================

def create_document(
    text: str,
    chunk_type: str,
    article_number: Optional[str] = None,
    article_title: Optional[str] = None,
    annex_number: Optional[str] = None,
    recital_number: Optional[str] = None
) -> Document:
    """
    Create a LangChain Document with legal metadata.
    创建带法律元数据的 LangChain Document。

    Metadata is critical for legal RAG because it helps locate Article,
    Annex, Recital, and page information.
    metadata 对法律 RAG 很关键，因为它能帮助定位 Article、Annex、Recital 和页码。
    """
    page_start, page_end = infer_pages(text)

    metadata = {
        "source": "EU AI Act",
        "chunk_type": chunk_type,
        "article_number": article_number,
        "article_title": article_title,
        "annex_number": annex_number,
        "recital_number": recital_number,
        "page_start": page_start,
        "page_end": page_end,
    }

    return Document(
        page_content=text.strip(),
        metadata=metadata
    )

In [8]:
# ==============================
# Module 1.7: Split recitals and front matter
# 模块 1.7：切分 Recital 和前置文本
# ==============================

def split_recitals_or_frontmatter(text: str) -> List[Document]:
    """
    Split front matter and recitals.
    切分前置文本和 Recitals。

    Recitals explain legislative purpose, background, and interpretation.
    Recital 通常解释立法目的、背景和解释原则。
    """
    docs = []

    recital_matches = list(RECITAL_PATTERN.finditer(text))

    # If no recital is found, keep it as front matter.
    # 如果没有发现 recital，则作为 front matter 保存。
    if not recital_matches:
        if len(text.strip()) > 300:
            docs.append(
                create_document(
                    text=text,
                    chunk_type="frontmatter_or_recital"
                )
            )

        return docs

    # Text before the first recital.
    # 第一个 recital 之前的文本。
    if recital_matches[0].start() > 0:
        pre_text = text[:recital_matches[0].start()].strip()

        if len(pre_text) > 300:
            docs.append(
                create_document(
                    text=pre_text,
                    chunk_type="frontmatter"
                )
            )

    # Split each recital.
    # 切分每一条 recital。
    for i, match in enumerate(recital_matches):
        start = match.start()
        end = recital_matches[i + 1].start() if i + 1 < len(recital_matches) else len(text)

        recital_text = text[start:end].strip()
        recital_number = match.group(1)

        if len(recital_text) > 100:
            docs.append(
                create_document(
                    text=recital_text,
                    chunk_type="recital",
                    recital_number=recital_number
                )
            )

    return docs

In [9]:
# ==============================
# Module 1.8: Legal-structure-aware splitting
# 模块 1.8：基于法律结构进行切分
# ==============================

def split_legal_document(full_text: str) -> List[Document]:
    """
    Split the EU AI Act into legal-structure-aware chunks.
    将 EU AI Act 切分为法律结构感知 chunks。

    Main chunk types:
    主要 chunk 类型：
    - frontmatter
    - recital
    - article
    - annex
    """
    docs = []

    # Step 1: Split out Annex sections first.
    # 第一步：先切出 Annex，避免 Annex 被 Article 逻辑干扰。
    sections_by_annex = split_text_by_header_positions(
        full_text,
        ANNEX_HEADER_PATTERN
    )

    main_body_parts = []
    annex_parts = []

    for section in sections_by_annex:
        if ANNEX_HEADER_PATTERN.search(section[:500]):
            annex_parts.append(section)
        else:
            main_body_parts.append(section)

    main_body_text = "\n\n".join(main_body_parts)

    # Step 2: Split main body by Article.
    # 第二步：主体部分按 Article 切分。
    article_sections = split_text_by_header_positions(
        main_body_text,
        ARTICLE_HEADER_PATTERN
    )

    for section in article_sections:
        article_number = infer_article_number(section)

        if article_number:
            docs.append(
                create_document(
                    text=section,
                    chunk_type="article",
                    article_number=article_number,
                    article_title=infer_article_title(section)
                )
            )
        else:
            # Content before Article 1 usually contains front matter and recitals.
            # Article 1 之前通常包含前置文本和 Recitals。
            docs.extend(split_recitals_or_frontmatter(section))

    # Step 3: Add Annex sections.
    # 第三步：加入 Annex 部分。
    for annex_text in annex_parts:
        annex_number = infer_annex_number(annex_text)

        docs.append(
            create_document(
                text=annex_text,
                chunk_type="annex",
                annex_number=annex_number
            )
        )

    # Remove tiny fragments.
    # 删除过短碎片。
    docs = [
        doc for doc in docs
        if len(doc.page_content.strip()) > 120
    ]

    return docs


legal_docs = split_legal_document(full_text)

print(f"✅ Legal-level chunks created: {len(legal_docs)}")

print("\nChunk type distribution before secondary splitting:")
counter = Counter(doc.metadata.get("chunk_type") for doc in legal_docs)

for chunk_type, count in counter.items():
    print(f"{chunk_type}: {count}")

✅ Legal-level chunks created: 307

Chunk type distribution before secondary splitting:
frontmatter: 1
recital: 180
article: 113
annex: 13


In [10]:
# ==============================
# Module 1.9: Secondary splitting for long legal chunks
# 模块 1.9：对过长法律 chunk 做二级切分
# ==============================

secondary_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " "
    ]
)


def secondary_split_documents(docs: List[Document]) -> List[Document]:
    """
    Split long legal chunks into smaller retrieval chunks.
    将过长的法律 chunk 切成更适合检索的小 chunk。

    Original legal metadata is copied to every subchunk.
    原始法律 metadata 会复制到每个子 chunk。
    """
    final_docs = []

    for parent_idx, doc in enumerate(docs):
        text = doc.page_content
        base_metadata = dict(doc.metadata)

        # Keep relatively short chunks unchanged.
        # 较短 chunk 保持不变。
        if len(text) <= CHUNK_SIZE * 1.3:
            metadata = dict(base_metadata)
            metadata["parent_id"] = parent_idx
            metadata["subchunk_id"] = 0
            metadata["is_subchunk"] = False

            final_docs.append(
                Document(
                    page_content=text,
                    metadata=metadata
                )
            )

        else:
            # Split long chunks with overlap.
            # 对长 chunk 做带 overlap 的切分。
            sub_texts = secondary_splitter.split_text(text)

            for sub_idx, sub_text in enumerate(sub_texts):
                metadata = dict(base_metadata)
                metadata["parent_id"] = parent_idx
                metadata["subchunk_id"] = sub_idx
                metadata["is_subchunk"] = True
                metadata["parent_chunk_type"] = base_metadata.get("chunk_type")

                final_docs.append(
                    Document(
                        page_content=sub_text,
                        metadata=metadata
                    )
                )

    return final_docs


final_docs = secondary_split_documents(legal_docs)

print(f"✅ Final retrieval chunks created: {len(final_docs)}")

print("\nChunk type distribution after secondary splitting:")
counter = Counter(doc.metadata.get("chunk_type") for doc in final_docs)

for chunk_type, count in counter.items():
    print(f"{chunk_type}: {count}")

✅ Final retrieval chunks created: 522

Chunk type distribution after secondary splitting:
frontmatter: 3
recital: 231
article: 250
annex: 38


In [11]:
# ==============================
# Module 1.10: Save chunks to JSONL
# 模块 1.10：保存 chunks 到 JSONL
# ==============================

def save_documents_to_jsonl(docs: List[Document], output_path: str) -> None:
    """
    Save documents to JSONL for reproducibility and debugging.
    将 documents 保存为 JSONL，方便复现和调试。
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for idx, doc in enumerate(docs):
            record = {
                "chunk_id": idx,
                "page_content": doc.page_content,
                "metadata": doc.metadata
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


save_documents_to_jsonl(final_docs, CHUNK_OUTPUT_PATH)

print(f"✅ Chunks saved to: {CHUNK_OUTPUT_PATH}")
print(f"✅ Number of chunks saved: {len(final_docs)}")

✅ Chunks saved to: chunks_legal_structure.jsonl
✅ Number of chunks saved: 522


In [12]:
# ==============================
# Module 1.11: Sanity check for important Articles
# 模块 1.11：检查关键 Article 是否能被识别
# ==============================

def find_chunks_by_article(article_number: str, docs: List[Document], limit: int = 3) -> None:
    """
    Print chunks belonging to a specific Article.
    打印属于某个 Article 的 chunks。
    """
    matched = [
        doc for doc in docs
        if str(doc.metadata.get("article_number")) == str(article_number)
    ]

    print(f"\nFound {len(matched)} chunks for Article {article_number}")

    for i, doc in enumerate(matched[:limit], start=1):
        print("\n" + "-" * 80)
        print(f"Article {article_number} chunk {i}")
        print("Metadata:", doc.metadata)
        print(doc.page_content[:800])


# Important Articles for RA-Gnarok.
# RA-Gnarok 中很可能用到的关键法条。
find_chunks_by_article("3", final_docs, limit=2)    # Definitions / 定义
find_chunks_by_article("5", final_docs, limit=2)    # Prohibited AI practices / 禁止性 AI 实践
find_chunks_by_article("99", final_docs, limit=2)   # Penalties / 处罚


Found 13 chunks for Article 3

--------------------------------------------------------------------------------
Article 3 chunk 1
Metadata: {'source': 'EU AI Act', 'chunk_type': 'article', 'article_number': '3', 'article_title': 'Definitions', 'annex_number': None, 'recital_number': None, 'page_start': 167, 'page_end': 180, 'parent_id': 183, 'subchunk_id': 0, 'is_subchunk': True, 'parent_chunk_type': 'article'}
Article 3

Definitions

For the purposes of this Regulation, the following definitions apply:

(1) ‘AI system’ means a machine-based system designed to operate with varying levels of

autonomy, that may exhibit adaptiveness after deployment and that, for explicit or

implicit objectives, infers, from the input it receives, how to generate outputs such as

predictions, content, recommendations, or decisions that can influence physical or virtual

environments;

(2) ‘risk’ means the combination of the probability of an occurrence of harm and the

severity of that harm;

(3) ‘prov

In [13]:
# ==============================
# Module 2.1: Install local retrieval dependencies
# 模块 2.1：安装本地检索依赖
# ==============================

import sys

# FAISS: local dense vector search
# FAISS：本地稠密向量检索
!{sys.executable} -m pip install -qU faiss-cpu

# BM25: local keyword retrieval
# BM25：本地关键词检索
!{sys.executable} -m pip install -qU rank_bm25

# LangChain retrievers and vector stores
# LangChain 检索器和向量库
!{sys.executable} -m pip install -qU langchain-community langchain-huggingface

# Local embedding model support
# 本地 embedding 模型支持
!{sys.executable} -m pip install -qU sentence-transformers

print("✅ Module 2 dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gcsfs 2025.3.0 requires fsspe

In [14]:
# ==============================
# Module 2.2: Imports and load chunks
# 模块 2.2：导入库并加载 chunks
# ==============================

import os
import re
import json
from typing import List, Dict, Tuple, Any, Optional

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings


# ------------------------------
# Paths
# 路径配置
# ------------------------------

CHUNK_OUTPUT_PATH = "chunks_legal_structure.jsonl"
FAISS_INDEX_PATH = "faiss_eu_ai_act_index"


def load_documents_from_jsonl(path: str) -> List[Document]:
    """
    Load LangChain Documents from JSONL.
    从 JSONL 文件恢复 LangChain Document。

    This is useful if final_docs is not in memory.
    如果 final_docs 不在内存中，可以从文件恢复。
    """
    docs = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            docs.append(
                Document(
                    page_content=record["page_content"],
                    metadata=record["metadata"]
                )
            )

    return docs


# Try to use final_docs from Module 1.
# 优先使用 Module 1 中已经生成的 final_docs。
try:
    final_docs
    print(f"✅ final_docs found in memory. Number of chunks: {len(final_docs)}")

except NameError:
    print("⚠️ final_docs not found in memory. Loading from JSONL...")

    if not os.path.exists(CHUNK_OUTPUT_PATH):
        raise FileNotFoundError(
            "chunks_legal_structure.jsonl not found. Please run Module 1 first."
        )

    final_docs = load_documents_from_jsonl(CHUNK_OUTPUT_PATH)
    print(f"✅ Loaded final_docs from JSONL. Number of chunks: {len(final_docs)}")


if len(final_docs) == 0:
    raise ValueError("final_docs is empty. Please check Module 1 output.")

✅ final_docs found in memory. Number of chunks: 522


In [15]:
# ==============================
# Module 2.3: Initialize local embedding model
# 模块 2.3：初始化本地 Embedding 模型
# ==============================

# Recommended lightweight local embedding model.
# 推荐使用轻量级本地 embedding 模型。
#
# Advantages:
# 优点：
# - Small and fast
# - No API required
# - Good baseline for semantic retrieval
#
# 注意：
# If Kaggle internet is disabled, you need to add this model as a Kaggle Dataset.
# 如果 Kaggle 关闭网络，需要提前把模型作为 Kaggle Dataset 加入。
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"🔧 Loading local embedding model: {EMBEDDING_MODEL_NAME}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    encode_kwargs={
        # Normalize embeddings for more stable similarity search.
        # 归一化 embedding，使相似度检索更稳定。
        "normalize_embeddings": True
    }
)

print("✅ Local embedding model loaded.")

🔧 Loading local embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Local embedding model loaded.


In [16]:
# ==============================
# Module 2.4: Build FAISS dense vector index
# 模块 2.4：构建 FAISS 稠密向量索引
# ==============================

def build_or_load_faiss_index(
    docs: List[Document],
    embeddings,
    index_path: str = FAISS_INDEX_PATH,
    rebuild: bool = False
):
    """
    Build or load FAISS vector store.
    构建或加载 FAISS 向量库。

    Args:
        docs: retrieval chunks / 检索 chunks
        embeddings: local embedding model / 本地 embedding 模型
        index_path: local FAISS index path / 本地 FAISS 索引路径
        rebuild: whether to force rebuild / 是否强制重建
    """
    if os.path.exists(index_path) and not rebuild:
        print(f"📦 Loading existing FAISS index from: {index_path}")

        vectorstore = FAISS.load_local(
            index_path,
            embeddings,
            allow_dangerous_deserialization=True
        )

        print("✅ FAISS index loaded.")
        return vectorstore

    print("🚀 Building FAISS index from final_docs...")

    vectorstore = FAISS.from_documents(
        final_docs,
        embeddings
    )

    vectorstore.save_local(index_path)

    print(f"✅ FAISS index built and saved to: {index_path}")
    return vectorstore


# If you changed Module 1 chunking, set rebuild=True.
# 如果你修改了 Module 1 的切分逻辑，请设置 rebuild=True。
vectorstore = build_or_load_faiss_index(
    docs=final_docs,
    embeddings=embeddings,
    index_path=FAISS_INDEX_PATH,
    rebuild=False
)


# Dense retriever.
# 稠密语义检索器。
dense_retriever = vectorstore.as_retriever(
    search_kwargs={
        # k=10 is safer than k=3 for legal QA.
        # 法律问答中 k=3 太少，k=10 更稳。
        "k": 10
    }
)

print("✅ Dense FAISS retriever ready.")

🚀 Building FAISS index from final_docs...
✅ FAISS index built and saved to: faiss_eu_ai_act_index
✅ Dense FAISS retriever ready.


In [17]:
# ==============================
# Module 2.5: Build BM25 sparse keyword index
# 模块 2.5：构建 BM25 关键词索引
# ==============================

print("🚀 Building BM25 retriever from final_docs...")

bm25_retriever = BM25Retriever.from_documents(final_docs)

# Number of BM25 candidates.
# BM25 返回候选 chunk 的数量。
bm25_retriever.k = 10

print("✅ BM25 sparse retriever ready.")

🚀 Building BM25 retriever from final_docs...
✅ BM25 sparse retriever ready.


In [18]:
# ==============================
# Module 2.6: Deduplication and RRF fusion
# 模块 2.6：去重与 RRF 融合排序
# ==============================

def make_doc_key(doc: Document) -> Tuple:
    """
    Create a stable key for document deduplication.
    为文档创建稳定 key，用于去重。

    We prioritize legal metadata, then use text prefix as fallback.
    优先使用法律 metadata，其次使用文本前缀兜底。
    """
    meta = doc.metadata or {}

    return (
        meta.get("chunk_type"),
        meta.get("article_number"),
        meta.get("annex_number"),
        meta.get("recital_number"),
        meta.get("parent_id"),
        meta.get("subchunk_id"),
        doc.page_content[:120]
    )


def reciprocal_rank_fusion(
    result_lists: List[List[Document]],
    rrf_k: int = 60,
    top_n: int = 12
) -> List[Document]:
    """
    Fuse multiple ranked lists using Reciprocal Rank Fusion.
    使用 Reciprocal Rank Fusion 融合多个检索结果列表。

    Formula:
    公式：
        score += 1 / (rrf_k + rank)

    Why RRF:
    为什么用 RRF：
    - Simple and robust
      简单且稳定
    - Works well for combining dense and sparse retrieval
      适合融合向量检索和关键词检索
    """
    scores = {}
    doc_store = {}

    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            key = make_doc_key(doc)

            if key not in doc_store:
                doc_store[key] = doc

            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank)

    ranked_keys = sorted(
        scores.keys(),
        key=lambda key: scores[key],
        reverse=True
    )

    fused_docs = [
        doc_store[key]
        for key in ranked_keys[:top_n]
    ]

    return fused_docs


print("✅ RRF fusion function ready.")

✅ RRF fusion function ready.


In [19]:
# ==============================
# Module 2.7: Metadata boost for Article and Annex
# 模块 2.7：Article 和 Annex 元数据增强
# ==============================

def extract_article_mentions(query: str) -> List[str]:
    """
    Extract Article numbers mentioned in the query.
    从 query 中提取 Article 编号。

    Example:
    示例：
    "What does Article 99(5) say?" -> ["99"]
    """
    matches = re.findall(
        r"Article\s+(\d+[A-Za-z]?)",
        query,
        flags=re.IGNORECASE
    )

    return list(dict.fromkeys(matches))


def extract_annex_mentions(query: str) -> List[str]:
    """
    Extract Annex numbers mentioned in the query.
    从 query 中提取 Annex 编号。

    Example:
    示例：
    "What does Annex III say?" -> ["III"]
    """
    matches = re.findall(
        r"Annex\s+([IVXLC]+)",
        query,
        flags=re.IGNORECASE
    )

    return list(dict.fromkeys([m.upper() for m in matches]))


def metadata_boost_docs(
    query: str,
    docs: List[Document],
    article_limit: int = 6,
    annex_limit: int = 6
) -> List[Document]:
    """
    Add documents that exactly match Article or Annex mentions.
    加入与问题中 Article 或 Annex 编号精确匹配的 chunks。

    This is important for legal QA because many questions explicitly mention Articles.
    这对法律问答很重要，因为很多问题会明确提到 Article。
    """
    boosted_docs = []

    article_numbers = extract_article_mentions(query)
    annex_numbers = extract_annex_mentions(query)

    # Boost Article chunks.
    # 增强 Article 精确匹配。
    for article_no in article_numbers:
        matched = [
            doc for doc in docs
            if str(doc.metadata.get("article_number")) == str(article_no)
        ]

        boosted_docs.extend(matched[:article_limit])

    # Boost Annex chunks.
    # 增强 Annex 精确匹配。
    for annex_no in annex_numbers:
        matched = [
            doc for doc in docs
            if str(doc.metadata.get("annex_number")).upper() == str(annex_no).upper()
        ]

        boosted_docs.extend(matched[:annex_limit])

    return boosted_docs


print("✅ Metadata boost functions ready.")

✅ Metadata boost functions ready.


In [20]:
# ==============================
# Module 2.8: Local hybrid retriever
# 模块 2.8：本地混合检索器
# ==============================

def hybrid_retrieve(
    query: str,
    top_n: int = 12,
    dense_k: int = 10,
    bm25_k: int = 10,
    use_metadata_boost: bool = True
) -> List[Document]:
    """
    Local hybrid retrieval:
    本地混合检索：

    1. FAISS dense semantic retrieval
       FAISS 稠密语义检索

    2. BM25 sparse keyword retrieval
       BM25 稀疏关键词检索

    3. Metadata boost for Article / Annex
       Article / Annex 元数据增强

    4. RRF fusion
       RRF 融合排序
    """
    if not query or not str(query).strip():
        return []

    query = str(query).strip()

    # Adjust candidate sizes.
    # 调整候选数量。
    dense_retriever.search_kwargs["k"] = dense_k
    bm25_retriever.k = bm25_k

    # Dense semantic retrieval.
    # 稠密语义检索。
    dense_docs = dense_retriever.invoke(query)

    # Sparse keyword retrieval.
    # 稀疏关键词检索。
    bm25_docs = bm25_retriever.invoke(query)

    result_lists = [
        dense_docs,
        bm25_docs
    ]

    # Metadata boost.
    # 元数据增强。
    if use_metadata_boost:
        boosted_docs = metadata_boost_docs(
            query=query,
            docs=final_docs,
            article_limit=6,
            annex_limit=6
        )

        if boosted_docs:
            result_lists.append(boosted_docs)

    # Fuse results.
    # 融合检索结果。
    fused_docs = reciprocal_rank_fusion(
        result_lists=result_lists,
        rrf_k=60,
        top_n=top_n
    )

    return fused_docs


print("✅ Local hybrid retriever ready.")

✅ Local hybrid retriever ready.


In [21]:
# ==============================
# Module 2.9: Format retrieved context
# 模块 2.9：格式化检索上下文
# ==============================

def format_doc_label(doc: Document, index: int) -> str:
    """
    Create a readable label for a retrieved chunk.
    为检索到的 chunk 创建可读标签。
    """
    meta = doc.metadata or {}

    parts = [f"[Chunk {index}]"]

    if meta.get("article_number"):
        parts.append(f"Article {meta.get('article_number')}")

    if meta.get("article_title"):
        parts.append(f"- {meta.get('article_title')}")

    if meta.get("annex_number"):
        parts.append(f"Annex {meta.get('annex_number')}")

    if meta.get("recital_number"):
        parts.append(f"Recital {meta.get('recital_number')}")

    if meta.get("page_start"):
        parts.append(f"Pages {meta.get('page_start')}-{meta.get('page_end')}")

    return " ".join(parts)


def format_retrieved_context(docs: List[Document]) -> str:
    """
    Format retrieved documents into a context string.
    将检索结果格式化成上下文字符串。

    This will be used by the local answer generator later.
    后续本地答案生成器会使用这个 context。
    """
    blocks = []

    for idx, doc in enumerate(docs, start=1):
        label = format_doc_label(doc, idx)
        block = f"{label}\n{doc.page_content}"
        blocks.append(block)

    return "\n\n".join(blocks)


def docs_to_debug_records(docs: List[Document]) -> List[Dict[str, Any]]:
    """
    Convert retrieved docs to debug records.
    将检索结果转换成 debug 记录。

    Useful for checkpoint logs and error analysis.
    方便之后保存日志和做错误分析。
    """
    records = []

    for idx, doc in enumerate(docs, start=1):
        records.append({
            "rank": idx,
            "metadata": doc.metadata,
            "preview": doc.page_content[:500]
        })

    return records


print("✅ Context formatting functions ready.")

✅ Context formatting functions ready.


In [22]:
# ==============================
# Module 2.10: Retrieval sanity tests
# 模块 2.10：检索质量快速测试
# ==============================

test_queries = [
    "What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?",
    "How does the EU AI Act define AI system in Article 3?",
    "What are the obligations for providers of general-purpose AI models?",
    "What does Annex III say about high-risk AI systems?"
]

for query in test_queries:
    print("\n" + "=" * 120)
    print(f"🔍 Test query / 测试问题:\n{query}")

    retrieved_docs = hybrid_retrieve(
        query=query,
        top_n=5,
        dense_k=10,
        bm25_k=10,
        use_metadata_boost=True
    )

    for i, doc in enumerate(retrieved_docs, start=1):
        print("\n" + "-" * 90)
        print(format_doc_label(doc, i))
        print(doc.page_content[:800])


🔍 Test query / 测试问题:
What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?

------------------------------------------------------------------------------------------
[Chunk 1] Article 99 - Penalties Pages 391-395
Article 99

Penalties

1. In compliance with the terms and conditions laid down in this Regulation, Member States

shall lay down the rules on penalties and other enforcement measures, which may also

include warnings and non-monetary measures, applicable to infringements of this

Regulation by operators, and shall take all measures necessary to ensure that they are

properly and effectively implemented and taking into account the guidelines issued by

the Commission pursuant to Article 96. The penalties provided for shall be effective,

proportionate and dissuasive. They shall take into ▌ account the interests of SMEs,

including start-ups, and their economic viability.


[PAGE 391]
2. The Member States sha

In [23]:
# ==============================
# Module 3.1: Imports and configuration
# 模块 3.1：导入库与基础配置
# ==============================

import re
from typing import List, Dict, Any

# This module is fully local.
# 本模块完全本地运行，不调用 Gemini 或任何外部 API。
USE_LLM_REWRITE = False

print("✅ Module 3 loaded: Local multi-query retriever mode enabled.")
print("✅ No Gemini API will be used.")

✅ Module 3 loaded: Local multi-query retriever mode enabled.
✅ No Gemini API will be used.


In [24]:
# ==============================
# Module 3.2: Rule-based question router
# 模块 3.2：规则型问题路由器
# ==============================

def classify_question_type(question: str) -> str:
    """
    Classify the question type using rule-based patterns.
    使用规则模式判断问题类型。

    Why:
    为什么需要分类：
    Different question types need different retrieval hints.
    不同问题类型需要不同的检索提示。
    """
    q = question.lower().strip()

    # Article-specific questions.
    # 明确提到 Article 的问题。
    if re.search(r"\barticle\s+\d+[a-z]?\b", q):
        return "article_specific"

    # Annex-specific questions.
    # 明确提到 Annex 的问题。
    if re.search(r"\bannex\s+[ivxlc]+\b", q):
        return "annex_specific"

    # Definition questions.
    # 定义类问题。
    if any(phrase in q for phrase in [
        "define",
        "definition",
        "what is meant by",
        "meaning of",
        "how does the act define",
        "how is",
        "what does the term"
    ]):
        return "definition"

    # Penalty / fine questions.
    # 处罚 / 罚款类问题。
    if any(term in q for term in [
        "fine",
        "penalty",
        "administrative fine",
        "turnover",
        "eur",
        "€",
        "non-compliance",
        "infringement"
    ]):
        return "penalty"

    # General-purpose AI / GPAI questions.
    # 通用人工智能模型相关问题。
    if any(term in q for term in [
        "general-purpose",
        "general purpose",
        "gpai",
        "systemic risk",
        "high-impact capabilities",
        "flop",
        "flops"
    ]):
        return "gpai"

    # High-risk AI questions.
    # 高风险 AI 系统相关问题。
    if any(term in q for term in [
        "high-risk",
        "high risk",
        "high risk ai",
        "high-risk ai"
    ]):
        return "high_risk"

    # Prohibited AI practices.
    # 禁止性 AI 实践相关问题。
    if any(term in q for term in [
        "prohibited",
        "prohibition",
        "banned",
        "forbidden",
        "article 5"
    ]):
        return "prohibited_practices"

    # Yes / No questions.
    # 判断题 / 是非题。
    if q.startswith((
        "is ",
        "are ",
        "does ",
        "do ",
        "can ",
        "could ",
        "must ",
        "should ",
        "would "
    )):
        return "yes_no"

    # Scenario-based questions.
    # 场景推理题。
    if any(term in q for term in [
        "if ",
        "scenario",
        "company",
        "provider",
        "deployer",
        "authority",
        "wants to",
        "plans to",
        "uses an ai system",
        "develops an ai system"
    ]):
        return "scenario"

    return "general"


def get_router_hints(question_type: str) -> List[str]:
    """
    Return extra retrieval hints based on question type.
    根据问题类型返回额外检索提示。

    These hints are used as additional search queries.
    这些提示会作为额外检索 query。
    """
    hints = []

    if question_type == "definition":
        hints.extend([
            "Article 3 definitions EU AI Act",
            "means definition term AI system provider deployer AI Office"
        ])

    elif question_type == "penalty":
        hints.extend([
            "Article 99 penalties administrative fines non-compliance turnover EUR",
            "Article 100 fines Union institutions bodies offices agencies"
        ])

    elif question_type == "gpai":
        hints.extend([
            "Chapter V general-purpose AI models systemic risk Article 51 Article 52 Article 53 Article 55",
            "general-purpose AI model systemic risk FLOPs high-impact capabilities provider obligations"
        ])

    elif question_type == "high_risk":
        hints.extend([
            "Article 6 high-risk AI systems classification Annex III",
            "Chapter III high-risk AI systems requirements obligations conformity assessment"
        ])

    elif question_type == "prohibited_practices":
        hints.extend([
            "Article 5 prohibited AI practices prohibition banned AI systems",
            "unacceptable risk prohibited practices biometric social scoring manipulation"
        ])

    elif question_type == "annex_specific":
        hints.extend([
            "Annex EU AI Act high-risk AI systems categories requirements",
            "Annex III high-risk AI systems"
        ])

    elif question_type == "yes_no":
        hints.extend([
            "conditions exceptions obligations scope applicability EU AI Act",
            "prohibited permitted exception unless subject to conditions"
        ])

    elif question_type == "scenario":
        hints.extend([
            "provider deployer obligations conditions exceptions compliance EU AI Act",
            "requirements obligations prohibited high-risk general-purpose AI system"
        ])

    return hints


print("✅ Question router ready.")

✅ Question router ready.


In [25]:
# ==============================
# Module 3.3: Rule-based legal keyword extraction
# 模块 3.3：规则型法律关键词抽取
# ==============================

def clean_query_text(text: str) -> str:
    """
    Clean query text.
    清洗 query 文本。
    """
    text = str(text).strip()
    text = text.strip('"').strip("'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_legal_keywords(question: str) -> str:
    """
    Extract important legal keywords from the question.
    从问题中抽取重要法律关键词。

    This replaces LLM-based query rewriting.
    这一步替代原来的 LLM query rewriting。
    """
    q = str(question)
    keywords = []

    # Article mentions, including paragraph references.
    # Article 编号，包括段落引用。
    article_mentions = re.findall(
        r"Article\s+\d+[A-Za-z]?(?:\(\d+\))?(?:\([a-z]\))?",
        q,
        flags=re.IGNORECASE
    )
    keywords.extend(article_mentions)

    # Annex mentions.
    # Annex 编号。
    annex_mentions = re.findall(
        r"Annex\s+[IVXLC]+",
        q,
        flags=re.IGNORECASE
    )
    keywords.extend(annex_mentions)

    # Monetary amounts and percentages.
    # 金额和百分比。
    amount_mentions = re.findall(
        r"(?:EUR|€)\s*\d[\d\s,\.]*|\d+\s*%",
        q,
        flags=re.IGNORECASE
    )
    keywords.extend(amount_mentions)

    # Dates.
    # 日期。
    date_mentions = re.findall(
        r"\b\d{1,2}\s+[A-Z][a-z]+\s+\d{4}\b|\b\d{4}\b",
        q
    )
    keywords.extend(date_mentions)

    # FLOP / computation threshold.
    # FLOP / 计算量阈值。
    flop_mentions = re.findall(
        r"\d+\^?\d*\s*FLOPs?|\d+\s*\*\*\s*\d+\s*FLOPs?|FLOPs?",
        q,
        flags=re.IGNORECASE
    )
    keywords.extend(flop_mentions)

    # Important legal terms in the EU AI Act.
    # EU AI Act 中的重要法律术语。
    legal_terms = [
        "AI system",
        "high-risk AI system",
        "high-risk AI systems",
        "general-purpose AI model",
        "general-purpose AI models",
        "systemic risk",
        "high-impact capabilities",
        "AI Office",
        "provider",
        "providers",
        "deployer",
        "deployers",
        "importer",
        "importers",
        "distributor",
        "distributors",
        "authorised representative",
        "notified body",
        "notified bodies",
        "national competent authority",
        "market surveillance authority",
        "conformity assessment",
        "technical documentation",
        "transparency obligations",
        "AI literacy",
        "prohibited AI practices",
        "remote biometric identification",
        "real-time remote biometric identification",
        "biometric categorisation",
        "emotion recognition",
        "deep fake",
        "serious incident",
        "administrative fines",
        "penalties",
        "SMEs",
        "start-ups",
        "training data",
        "risk management system",
        "post-market monitoring",
        "fundamental rights impact assessment",
        "EU declaration of conformity",
        "CE marking"
    ]

    q_lower = q.lower()

    for term in legal_terms:
        if term.lower() in q_lower:
            keywords.append(term)

    # Add remaining meaningful words.
    # 加入剩余有意义词。
    stopwords = {
        "what", "which", "when", "where", "who", "whom", "whose",
        "is", "are", "was", "were", "be", "been", "being",
        "does", "do", "did", "can", "could", "should", "would", "must",
        "the", "a", "an", "of", "to", "in", "on", "for", "with", "by",
        "under", "about", "from", "and", "or", "as", "at", "into",
        "how", "why", "it", "its", "their", "they", "this", "that",
        "eu", "act", "ai", "article", "annex", "regulation"
    }

    words = re.findall(r"[A-Za-z][A-Za-z\-]+|\d+", q)

    for word in words:
        if word.lower() not in stopwords and len(word) > 2:
            keywords.append(word)

    # Deduplicate while preserving order.
    # 去重并保留顺序。
    seen = set()
    deduped = []

    for item in keywords:
        item = clean_query_text(item)
        key = item.lower()

        if item and key not in seen:
            deduped.append(item)
            seen.add(key)

    return " ".join(deduped)


print("✅ Legal keyword extractor ready.")

✅ Legal keyword extractor ready.


In [26]:
# ==============================
# Module 3.4: Rule-based pseudo-HyDE query
# 模块 3.4：规则型伪 HyDE Query
# ==============================

def build_rule_based_hyde_query(question: str, question_type: str) -> str:
    """
    Build a pseudo-HyDE query without using any LLM.
    不使用任何 LLM，构建规则型伪 HyDE query。

    Real HyDE uses an LLM to generate a hypothetical answer/document.
    真正的 HyDE 会用 LLM 生成假设性答案或文档。

    Here we use legal templates instead.
    这里我们用法律模板替代。
    """
    keyword_query = extract_legal_keywords(question)

    if question_type == "definition":
        return (
            "Article 3 definitions meaning definition term means "
            + keyword_query
        )

    if question_type == "penalty":
        return (
            "Article 99 Article 100 penalties administrative fines turnover EUR "
            "non-compliance infringement "
            + keyword_query
        )

    if question_type == "gpai":
        return (
            "Chapter V general-purpose AI model systemic risk high-impact capabilities "
            "FLOPs provider obligations Article 51 Article 52 Article 53 Article 55 "
            + keyword_query
        )

    if question_type == "high_risk":
        return (
            "Chapter III high-risk AI system Article 6 Annex III classification "
            "requirements obligations conformity assessment "
            + keyword_query
        )

    if question_type == "prohibited_practices":
        return (
            "Article 5 prohibited AI practices unacceptable risk prohibition banned "
            "manipulation biometric social scoring remote biometric identification "
            + keyword_query
        )

    if question_type == "annex_specific":
        return (
            "Annex EU AI Act categories requirements high-risk systems classification "
            + keyword_query
        )

    if question_type == "article_specific":
        return (
            "EU AI Act legal provision paragraph requirements obligations conditions exceptions "
            + keyword_query
        )

    if question_type == "yes_no":
        return (
            "EU AI Act conditions exceptions prohibition obligation scope applicability "
            "permitted prohibited unless subject to conditions "
            + keyword_query
        )

    if question_type == "scenario":
        return (
            "EU AI Act provider deployer obligations conditions exceptions compliance scenario "
            "requirements prohibited high-risk general-purpose AI system "
            + keyword_query
        )

    return (
        "EU AI Act legal provision requirements obligations conditions exceptions "
        + keyword_query
    )


print("✅ Rule-based pseudo-HyDE query builder ready.")

✅ Rule-based pseudo-HyDE query builder ready.


In [27]:
# ==============================
# Module 3.5: Build multi-query pack
# 模块 3.5：构建多路检索 query 包
# ==============================

def build_multi_queries(question: str, use_hyde: bool = True) -> Dict[str, Any]:
    """
    Build multiple retrieval queries locally.
    本地构建多个检索 query。

    Query types:
    query 类型：
    1. original_query: keeps the original intent
       原始问题：保留用户真实意图

    2. keyword_query: improves BM25 exact matching
       关键词 query：增强 BM25 精确匹配

    3. hyde_query: rule-based pseudo-HyDE for semantic recall
       伪 HyDE query：用于增强语义召回

    4. router_hints: question-type-specific hints
       路由提示：根据问题类型补充检索方向
    """
    question = str(question).strip()
    question_type = classify_question_type(question)

    original_query = clean_query_text(question)
    keyword_query = extract_legal_keywords(question)

    if use_hyde:
        hyde_query = build_rule_based_hyde_query(
            question=question,
            question_type=question_type
        )
    else:
        hyde_query = original_query

    router_hints = get_router_hints(question_type)

    query_pack = {
        "question_type": question_type,
        "original_query": original_query,
        "keyword_query": keyword_query,
        "hyde_query": hyde_query,
        "router_hints": router_hints,
        "use_llm_rewrite": False
    }

    return query_pack


# Quick test.
# 快速测试。
test_question = "What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?"

query_pack = build_multi_queries(test_question, use_hyde=True)

print("✅ Multi-query pack example:")
for key, value in query_pack.items():
    print(f"\n{key}:")
    print(value)

✅ Multi-query pack example:

question_type:
article_specific

original_query:
What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?

keyword_query:
Article 5 maximum administrative fine non-compliance prohibition practices referred

hyde_query:
EU AI Act legal provision paragraph requirements obligations conditions exceptions Article 5 maximum administrative fine non-compliance prohibition practices referred

router_hints:
[]

use_llm_rewrite:
False


In [28]:
# ==============================
# Module 3.6: Multi-query hybrid retrieval
# 模块 3.6：多路本地混合检索
# ==============================

def multi_query_retrieve(
    question: str,
    use_hyde: bool = True,
    top_n_per_query: int = 10,
    final_top_n: int = 12,
    dense_k: int = 10,
    bm25_k: int = 10
) -> Dict[str, Any]:
    """
    Retrieve documents using local multi-query + hybrid retrieval.
    使用本地多路 query + hybrid retrieval 检索文档。

    This function depends on Module 2:
    该函数依赖 Module 2：
    - hybrid_retrieve()
    - reciprocal_rank_fusion()
    - docs_to_debug_records()
    """
    query_pack = build_multi_queries(
        question=question,
        use_hyde=use_hyde
    )

    retrieval_queries = [
        query_pack["original_query"],
        query_pack["keyword_query"],
        query_pack["hyde_query"],
    ]

    # Add router hints.
    # 加入问题类型路由提示。
    retrieval_queries.extend(query_pack.get("router_hints", []))

    # Remove duplicate and empty queries.
    # 去除重复和空 query。
    cleaned_queries = []
    seen = set()

    for q in retrieval_queries:
        q = clean_query_text(q)

        if q and q.lower() not in seen:
            cleaned_queries.append(q)
            seen.add(q.lower())

    result_lists = []
    retrieval_debug = []

    for q in cleaned_queries:
        docs = hybrid_retrieve(
            query=q,
            top_n=top_n_per_query,
            dense_k=dense_k,
            bm25_k=bm25_k,
            use_metadata_boost=True
        )

        result_lists.append(docs)

        retrieval_debug.append({
            "query": q,
            "num_docs": len(docs),
            "top_docs": docs_to_debug_records(docs[:3])
        })

    # Fuse results from all queries.
    # 融合所有 query 的检索结果。
    fused_docs = reciprocal_rank_fusion(
        result_lists=result_lists,
        rrf_k=60,
        top_n=final_top_n
    )

    return {
        "docs": fused_docs,
        "query_pack": query_pack,
        "retrieval_queries": cleaned_queries,
        "debug": retrieval_debug
    }


print("✅ Local multi-query retriever ready.")

✅ Local multi-query retriever ready.


In [29]:
# ==============================
# Module 3.7: Display retrieval results
# 模块 3.7：展示检索结果
# ==============================

def show_retrieval_result(
    result: Dict[str, Any],
    max_docs: int = 5,
    preview_chars: int = 700
) -> None:
    """
    Print retrieval results in a readable format.
    以可读形式打印检索结果。
    """
    print("=" * 120)

    print("Question type / 问题类型:")
    print(result["query_pack"]["question_type"])

    print("\nRetrieval queries / 检索 queries:")
    for i, q in enumerate(result["retrieval_queries"], start=1):
        print(f"{i}. {q}")

    print("\nFinal retrieved documents / 最终检索文档:")
    for i, doc in enumerate(result["docs"][:max_docs], start=1):
        print("\n" + "-" * 90)
        print(format_doc_label(doc, i))
        print(doc.page_content[:preview_chars])


print("✅ Retrieval display function ready.")

✅ Retrieval display function ready.


In [30]:
# ==============================
# Module 3.8: Sanity tests
# 模块 3.8：快速测试
# ==============================

test_questions = [
    "What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?",
    "How does the EU AI Act define AI system?",
    "Is real-time remote biometric identification completely banned?",
    "What are the obligations for providers of general-purpose AI models with systemic risk?",
    "What does Annex III say about high-risk AI systems?"
]

for q in test_questions:
    print("\n\n" + "#" * 120)
    print(f"Test question / 测试问题:\n{q}")

    result = multi_query_retrieve(
        question=q,
        use_hyde=True,
        top_n_per_query=8,
        final_top_n=8,
        dense_k=10,
        bm25_k=10
    )

    show_retrieval_result(
        result=result,
        max_docs=4,
        preview_chars=700
    )



########################################################################################################################
Test question / 测试问题:
What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?
Question type / 问题类型:
article_specific

Retrieval queries / 检索 queries:
1. What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?
2. Article 5 maximum administrative fine non-compliance prohibition practices referred
3. EU AI Act legal provision paragraph requirements obligations conditions exceptions Article 5 maximum administrative fine non-compliance prohibition practices referred

Final retrieved documents / 最终检索文档:

------------------------------------------------------------------------------------------
[Chunk 1] Article 101 - Fines for providers of general-purpose AI models Pages 399-400
general-purpose AI model with systemic risk and give it an oppo

In [31]:
# ==============================
# Module 4.1: Imports and configuration
# 模块 4.1：导入库与基础配置
# ==============================

import re
from typing import List, Dict, Any, Tuple

from langchain_core.documents import Document


# ------------------------------
# Evidence grading configuration
# 证据评分配置
# ------------------------------

# Maximum number of reflection rounds.
# 最大补充检索轮数。建议先设为 1，避免检索过慢。
MAX_REFLECTION_ROUNDS = 1

# Evidence score threshold.
# 证据充分性的判断阈值。
EVIDENCE_THRESHOLD = 0.55

# Final number of evidence chunks.
# 最终保留的证据 chunk 数量。
FINAL_EVIDENCE_TOP_N = 12


print("✅ Module 4 configuration loaded.")

✅ Module 4 configuration loaded.


In [32]:
# ==============================
# Module 4.2: Text utility functions
# 模块 4.2：文本处理工具函数
# ==============================

def normalize_text(text: str) -> str:
    """
    Normalize text for rule-based matching.
    标准化文本，方便规则匹配。
    """
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def simple_tokenize(text: str) -> List[str]:
    """
    Simple tokenizer for keyword matching.
    简单分词，用于关键词匹配。
    """
    text = normalize_text(text)
    return re.findall(r"[a-z][a-z\-]+|\d+", text)


def get_combined_doc_text(docs: List[Document]) -> str:
    """
    Combine all retrieved document texts.
    合并所有检索到的文档文本。
    """
    return "\n\n".join([doc.page_content for doc in docs])


def contains_any(text: str, terms: List[str]) -> bool:
    """
    Check whether text contains any term in the list.
    检查文本是否包含列表中的任意术语。
    """
    text_norm = normalize_text(text)

    for term in terms:
        if normalize_text(term) in text_norm:
            return True

    return False


def extract_question_articles(question: str) -> List[str]:
    """
    Extract Article numbers mentioned in the question.
    从问题中提取 Article 编号。
    """
    matches = re.findall(
        r"Article\s+(\d+[A-Za-z]?)",
        str(question),
        flags=re.IGNORECASE
    )
    return list(dict.fromkeys(matches))


def extract_question_annexes(question: str) -> List[str]:
    """
    Extract Annex numbers mentioned in the question.
    从问题中提取 Annex 编号。
    """
    matches = re.findall(
        r"Annex\s+([IVXLC]+)",
        str(question),
        flags=re.IGNORECASE
    )
    return list(dict.fromkeys([m.upper() for m in matches]))


def get_doc_article_numbers(docs: List[Document]) -> List[str]:
    """
    Extract Article numbers from retrieved document metadata.
    从检索文档的 metadata 中提取 Article 编号。
    """
    article_numbers = []

    for doc in docs:
        article_no = doc.metadata.get("article_number")
        if article_no is not None:
            article_numbers.append(str(article_no))

    return list(dict.fromkeys(article_numbers))


def get_doc_annex_numbers(docs: List[Document]) -> List[str]:
    """
    Extract Annex numbers from retrieved document metadata.
    从检索文档的 metadata 中提取 Annex 编号。
    """
    annex_numbers = []

    for doc in docs:
        annex_no = doc.metadata.get("annex_number")
        if annex_no is not None:
            annex_numbers.append(str(annex_no).upper())

    return list(dict.fromkeys(annex_numbers))


print("✅ Text utility functions ready.")

✅ Text utility functions ready.


In [33]:
# ==============================
# Module 4.3: Question evidence signal analysis
# 模块 4.3：问题证据信号分析
# ==============================

def analyze_question_signals(question: str, question_type: str) -> Dict[str, Any]:
    """
    Analyze what evidence is likely needed for the question.
    分析问题可能需要哪些证据。

    The result will guide grading and supplementary retrieval.
    结果会用于证据评分和补充检索。
    """
    q = normalize_text(question)

    signals = {
        "question_type": question_type,
        "mentioned_articles": extract_question_articles(question),
        "mentioned_annexes": extract_question_annexes(question),

        "requires_definition": False,
        "requires_penalty": False,
        "requires_gpai": False,
        "requires_high_risk": False,
        "requires_prohibited_practices": False,
        "requires_yes_no_exceptions": False,
        "requires_scenario_reasoning": False,
        "requires_exact_value": False,

        "extra_queries": []
    }

    # Definition-related questions.
    # 定义类问题。
    if question_type == "definition" or any(x in q for x in [
        "define",
        "definition",
        "meaning of",
        "what is meant by",
        "how does the act define",
        "what does the term"
    ]):
        signals["requires_definition"] = True
        signals["extra_queries"].append("Article 3 definitions means definition")

    # Penalty / fine questions.
    # 处罚 / 罚款类问题。
    if question_type == "penalty" or any(x in q for x in [
        "fine",
        "penalty",
        "administrative fine",
        "turnover",
        "non-compliance",
        "infringement"
    ]):
        signals["requires_penalty"] = True
        signals["requires_exact_value"] = True
        signals["extra_queries"].append("Article 99 penalties administrative fines turnover EUR")
        signals["extra_queries"].append("Article 100 fines Union institutions bodies offices agencies")

    # General-purpose AI questions.
    # 通用人工智能模型相关问题。
    if question_type == "gpai" or any(x in q for x in [
        "general-purpose",
        "general purpose",
        "gpai",
        "systemic risk",
        "flop",
        "flops",
        "high-impact capabilities"
    ]):
        signals["requires_gpai"] = True
        signals["extra_queries"].append(
            "Chapter V general-purpose AI model systemic risk Article 51 Article 52 Article 53 Article 55"
        )

    # High-risk AI questions.
    # 高风险 AI 系统相关问题。
    if question_type == "high_risk" or any(x in q for x in [
        "high-risk",
        "high risk",
        "annex iii"
    ]):
        signals["requires_high_risk"] = True
        signals["extra_queries"].append("Article 6 high-risk AI systems Annex III classification")
        signals["extra_queries"].append("Chapter III high-risk AI systems requirements obligations")

    # Prohibited AI practices.
    # 禁止性 AI 实践相关问题。
    if question_type == "prohibited_practices" or any(x in q for x in [
        "prohibited",
        "prohibition",
        "banned",
        "forbidden",
        "article 5"
    ]):
        signals["requires_prohibited_practices"] = True
        signals["extra_queries"].append("Article 5 prohibited AI practices unacceptable risk prohibition")

    # Yes / No questions often require exceptions or conditions.
    # 是非题通常需要检索例外条件或适用条件。
    if question_type == "yes_no" or q.startswith((
        "is ",
        "are ",
        "does ",
        "do ",
        "can ",
        "could ",
        "must ",
        "should "
    )):
        signals["requires_yes_no_exceptions"] = True
        signals["extra_queries"].append("conditions exceptions unless permitted prohibited obligation")

    # Scenario-based questions.
    # 场景推理题。
    if question_type == "scenario" or any(x in q for x in [
        "if ",
        "scenario",
        "company",
        "provider",
        "deployer",
        "authority",
        "wants to",
        "plans to"
    ]):
        signals["requires_scenario_reasoning"] = True
        signals["extra_queries"].append("provider deployer obligations conditions exceptions compliance")

    # Exact values: dates, amounts, percentages, thresholds.
    # 精确值：日期、金额、百分比、阈值。
    if re.search(
        r"date|when|eur|€|%|flop|flops|threshold|maximum|minimum|within|days?|weeks?|months?|years?",
        q
    ):
        signals["requires_exact_value"] = True

    # Explicit Article / Annex references.
    # 显式 Article / Annex 引用。
    for article_no in signals["mentioned_articles"]:
        signals["extra_queries"].append(f"Article {article_no} EU AI Act")

    for annex_no in signals["mentioned_annexes"]:
        signals["extra_queries"].append(f"Annex {annex_no} EU AI Act")

    # Deduplicate extra queries.
    # 去重补充 query。
    seen = set()
    deduped_queries = []

    for extra_q in signals["extra_queries"]:
        key = normalize_text(extra_q)

        if key not in seen:
            deduped_queries.append(extra_q)
            seen.add(key)

    signals["extra_queries"] = deduped_queries

    return signals


print("✅ Question signal analyzer ready.")

✅ Question signal analyzer ready.


In [34]:
# ==============================
# Module 4.4: Keyword coverage calculation
# 模块 4.4：关键词覆盖率计算
# ==============================

def calculate_keyword_coverage(question: str, docs: List[Document]) -> float:
    """
    Calculate keyword coverage between question and retrieved documents.
    计算问题关键词在检索文档中的覆盖率。
    """
    if not docs:
        return 0.0

    # Use Module 3 keyword extractor if available.
    # 如果 Module 3 的关键词抽取函数可用，则优先使用。
    try:
        keyword_text = extract_legal_keywords(question)
    except NameError:
        keyword_text = question

    query_tokens = simple_tokenize(keyword_text)

    stop_tokens = {
        "what", "which", "when", "where", "how",
        "does", "with", "under", "article", "annex",
        "regulation", "legal", "provision", "requirements",
        "obligations", "system", "systems"
    }

    query_tokens = [
        token for token in query_tokens
        if len(token) > 2 and token not in stop_tokens
    ]

    if not query_tokens:
        return 0.0

    doc_text = normalize_text(get_combined_doc_text(docs))

    matched = 0
    for token in query_tokens:
        if token in doc_text:
            matched += 1

    return matched / max(len(query_tokens), 1)


print("✅ Keyword coverage function ready.")

✅ Keyword coverage function ready.


In [35]:
# ==============================
# Module 4.5: Local evidence grading
# 模块 4.5：本地证据评分
# ==============================

def grade_evidence(
    question: str,
    docs: List[Document],
    query_pack: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Grade whether the retrieved documents are sufficient.
    评估检索到的文档是否足够回答问题。

    The score is heuristic and designed for legal RAG debugging.
    该分数是启发式规则分数，主要用于法律 RAG 调试。
    """
    question_type = query_pack.get("question_type", "general")
    signals = analyze_question_signals(question, question_type)

    combined_text = get_combined_doc_text(docs)
    combined_norm = normalize_text(combined_text)

    doc_articles = get_doc_article_numbers(docs)
    doc_annexes = get_doc_annex_numbers(docs)

    score_parts = {}
    missing_signals = []

    # 1. Keyword coverage.
    # 1. 关键词覆盖率。
    keyword_coverage = calculate_keyword_coverage(question, docs)
    score_parts["keyword_coverage"] = keyword_coverage

    # 2. Article match.
    # 2. Article 匹配。
    mentioned_articles = signals["mentioned_articles"]

    if mentioned_articles:
        article_match = any(article_no in doc_articles for article_no in mentioned_articles)

        # Some questions mention Article 5, but the answer may appear in Article 99.
        # 有些问题提到 Article 5，但答案可能在 Article 99，例如 Article 5 违规罚款。
        if signals["requires_penalty"] and ("99" in doc_articles or "100" in doc_articles):
            article_match = True

        score_parts["article_match"] = 1.0 if article_match else 0.0

        if not article_match:
            missing_signals.append(f"missing mentioned Article(s): {mentioned_articles}")
    else:
        score_parts["article_match"] = 0.5

    # 3. Annex match.
    # 3. Annex 匹配。
    mentioned_annexes = signals["mentioned_annexes"]

    if mentioned_annexes:
        annex_match = any(annex_no in doc_annexes for annex_no in mentioned_annexes)
        score_parts["annex_match"] = 1.0 if annex_match else 0.0

        if not annex_match:
            missing_signals.append(f"missing mentioned Annex(es): {mentioned_annexes}")
    else:
        score_parts["annex_match"] = 0.5

    # 4. Type-specific evidence.
    # 4. 问题类型相关证据。
    type_scores = []

    if signals["requires_definition"]:
        ok = (
            "3" in doc_articles
            or contains_any(combined_norm, ["means", "definition", "defined as"])
        )
        type_scores.append(1.0 if ok else 0.0)

        if not ok:
            missing_signals.append("definition evidence likely missing")

    if signals["requires_penalty"]:
        ok = (
            "99" in doc_articles
            or "100" in doc_articles
            or contains_any(combined_norm, ["administrative fines", "penalties", "turnover", "eur"])
        )
        type_scores.append(1.0 if ok else 0.0)

        if not ok:
            missing_signals.append("penalty or fine evidence likely missing")

    if signals["requires_gpai"]:
        ok = (
            any(article_no in doc_articles for article_no in ["51", "52", "53", "54", "55"])
            or contains_any(combined_norm, ["general-purpose ai model", "systemic risk", "flops"])
        )
        type_scores.append(1.0 if ok else 0.0)

        if not ok:
            missing_signals.append("general-purpose AI evidence likely missing")

    if signals["requires_high_risk"]:
        ok = (
            "6" in doc_articles
            or "III" in doc_annexes
            or contains_any(combined_norm, ["high-risk ai system", "annex iii", "classification"])
        )
        type_scores.append(1.0 if ok else 0.0)

        if not ok:
            missing_signals.append("high-risk AI evidence likely missing")

    if signals["requires_prohibited_practices"]:
        ok = (
            "5" in doc_articles
            or contains_any(combined_norm, ["prohibited ai practices", "prohibition", "unacceptable risk"])
        )
        type_scores.append(1.0 if ok else 0.0)

        if not ok:
            missing_signals.append("prohibited practices evidence likely missing")

    if signals["requires_yes_no_exceptions"]:
        ok = contains_any(combined_norm, [
            "except",
            "exception",
            "unless",
            "provided that",
            "shall not apply",
            "subject to",
            "conditions"
        ])
        type_scores.append(1.0 if ok else 0.4)

    if signals["requires_scenario_reasoning"]:
        ok = contains_any(combined_norm, [
            "provider",
            "deployer",
            "obligations",
            "requirements",
            "shall",
            "conditions",
            "except"
        ])
        type_scores.append(1.0 if ok else 0.4)

    if type_scores:
        score_parts["type_specific"] = sum(type_scores) / len(type_scores)
    else:
        score_parts["type_specific"] = 0.5

    # 5. Exact value signal.
    # 5. 精确数值信号。
    if signals["requires_exact_value"]:
        ok = bool(re.search(
            r"(eur|€|\d+\s*%|\d+\s*000|\d+\^\d+|10\^?\d+|within\s+\w+|days?|weeks?|months?|years?)",
            combined_norm,
            flags=re.IGNORECASE
        ))

        score_parts["exact_value_signal"] = 1.0 if ok else 0.0

        if not ok:
            missing_signals.append("exact value, date, or threshold likely missing")
    else:
        score_parts["exact_value_signal"] = 0.5

    # Weighted final score.
    # 加权最终分数。
    final_score = (
        0.30 * score_parts["keyword_coverage"]
        + 0.20 * score_parts["article_match"]
        + 0.15 * score_parts["annex_match"]
        + 0.25 * score_parts["type_specific"]
        + 0.10 * score_parts["exact_value_signal"]
    )

    is_sufficient = final_score >= EVIDENCE_THRESHOLD

    return {
        "is_sufficient": is_sufficient,
        "evidence_score": round(final_score, 4),
        "score_parts": score_parts,
        "missing_signals": missing_signals,
        "question_signals": signals,
        "doc_articles": doc_articles,
        "doc_annexes": doc_annexes,
        "num_docs": len(docs)
    }


print("✅ Local evidence grader ready.")

✅ Local evidence grader ready.


In [36]:
# ==============================
# Module 4.6: Supplementary query generation
# 模块 4.6：补充检索 Query 生成
# ==============================

def build_supplementary_queries(question: str, grade: Dict[str, Any]) -> List[str]:
    """
    Build supplementary retrieval queries based on missing evidence.
    根据缺失证据信号构建补充检索 query。
    """
    signals = grade.get("question_signals", {})
    supplementary_queries = list(signals.get("extra_queries", []))

    # Always keep original question as one fallback query.
    # 始终保留原问题作为兜底 query。
    supplementary_queries.append(question)

    missing = " ".join(grade.get("missing_signals", [])).lower()

    if "penalty" in missing or "fine" in missing:
        supplementary_queries.append("Article 99 administrative fines non-compliance turnover EUR")
        supplementary_queries.append("Article 100 administrative fines Union institutions bodies offices agencies")

    if "definition" in missing:
        supplementary_queries.append("Article 3 definitions means definition")

    if "general-purpose" in missing or "gpai" in missing:
        supplementary_queries.append("Article 51 Article 52 Article 53 Article 55 general-purpose AI model systemic risk")

    if "high-risk" in missing:
        supplementary_queries.append("Article 6 Annex III high-risk AI systems classification")

    if "prohibited" in missing:
        supplementary_queries.append("Article 5 prohibited AI practices unacceptable risk")

    if "exact value" in missing or "threshold" in missing:
        supplementary_queries.append("maximum threshold EUR percentage FLOPs within days weeks months")

    # Add explicit article and annex mentions.
    # 加入问题中显式提到的 Article 和 Annex。
    for article_no in signals.get("mentioned_articles", []):
        supplementary_queries.append(f"Article {article_no} EU AI Act provision")

    for annex_no in signals.get("mentioned_annexes", []):
        supplementary_queries.append(f"Annex {annex_no} EU AI Act")

    # Deduplicate.
    # 去重。
    seen = set()
    cleaned_queries = []

    for query in supplementary_queries:
        query = clean_query_text(query)
        key = normalize_text(query)

        if query and key not in seen:
            cleaned_queries.append(query)
            seen.add(key)

    return cleaned_queries


def retrieve_supplementary_docs(
    supplementary_queries: List[str],
    top_n_per_query: int = 12,
    dense_k: int = 14,
    bm25_k: int = 14
) -> List[List[Document]]:
    """
    Retrieve additional documents using supplementary queries.
    使用补充 query 检索更多文档。
    """
    result_lists = []

    for query in supplementary_queries:
        docs = hybrid_retrieve(
            query=query,
            top_n=top_n_per_query,
            dense_k=dense_k,
            bm25_k=bm25_k,
            use_metadata_boost=True
        )

        result_lists.append(docs)

    return result_lists


print("✅ Supplementary retrieval functions ready.")

✅ Supplementary retrieval functions ready.


In [37]:
# ==============================
# Module 4.7: Retrieval with evidence grading
# 模块 4.7：带证据评分的检索主函数
# ==============================

def retrieve_with_evidence_reflection(
    question: str,
    max_reflection_rounds: int = MAX_REFLECTION_ROUNDS,
    final_top_n: int = FINAL_EVIDENCE_TOP_N
) -> Dict[str, Any]:
    """
    Retrieve evidence with local grading and controlled supplementary retrieval.
    使用本地评分和受控补充检索获取证据。

    Workflow:
    流程：
    1. Initial multi-query retrieval
       初始多路检索

    2. Local evidence grading
       本地证据评分

    3. Supplementary retrieval if evidence is insufficient
       如果证据不足，则进行补充检索

    4. Fuse and re-grade
       融合结果并重新评分
    """
    # Initial retrieval from Module 3.
    # 使用 Module 3 进行初始检索。
    initial_result = multi_query_retrieve(
        question=question,
        use_hyde=True,
        top_n_per_query=10,
        final_top_n=final_top_n,
        dense_k=10,
        bm25_k=10
    )

    current_docs = initial_result["docs"]
    query_pack = initial_result["query_pack"]

    grade = grade_evidence(
        question=question,
        docs=current_docs,
        query_pack=query_pack
    )

    debug_rounds = [
        {
            "round": 0,
            "retrieval_queries": initial_result["retrieval_queries"],
            "grade": grade,
            "top_docs": docs_to_debug_records(current_docs[:5])
        }
    ]

    # Controlled supplementary retrieval.
    # 受控补充检索。
    for round_id in range(1, max_reflection_rounds + 1):
        if grade["is_sufficient"]:
            break

        supplementary_queries = build_supplementary_queries(
            question=question,
            grade=grade
        )

        supplementary_result_lists = retrieve_supplementary_docs(
            supplementary_queries=supplementary_queries,
            top_n_per_query=12,
            dense_k=14,
            bm25_k=14
        )

        # Fuse current docs and supplementary results.
        # 融合当前文档和补充检索结果。
        all_result_lists = [current_docs] + supplementary_result_lists

        current_docs = reciprocal_rank_fusion(
            result_lists=all_result_lists,
            rrf_k=60,
            top_n=final_top_n
        )

        grade = grade_evidence(
            question=question,
            docs=current_docs,
            query_pack=query_pack
        )

        debug_rounds.append(
            {
                "round": round_id,
                "supplementary_queries": supplementary_queries,
                "grade": grade,
                "top_docs": docs_to_debug_records(current_docs[:5])
            }
        )

    return {
        "question": question,
        "docs": current_docs,
        "query_pack": query_pack,
        "final_grade": grade,
        "reflection_rounds_used": len(debug_rounds) - 1,
        "debug_rounds": debug_rounds
    }


print("✅ Evidence-aware retrieval function ready.")

✅ Evidence-aware retrieval function ready.


In [38]:
# ==============================
# Module 4.8: Display evidence result
# 模块 4.8：展示证据检索结果
# ==============================

def show_evidence_result(
    result: Dict[str, Any],
    max_docs: int = 5,
    preview_chars: int = 700
) -> None:
    """
    Display retrieval and evidence grading result.
    展示检索和证据评分结果。
    """
    print("=" * 120)

    print("Question / 问题:")
    print(result["question"])

    print("\nQuestion type / 问题类型:")
    print(result["query_pack"]["question_type"])

    print("\nSupplementary retrieval rounds used / 使用的补充检索轮数:")
    print(result["reflection_rounds_used"])

    print("\nFinal evidence grade / 最终证据评分:")
    print(result["final_grade"])

    print("\nFinal evidence documents / 最终证据文档:")
    for i, doc in enumerate(result["docs"][:max_docs], start=1):
        print("\n" + "-" * 90)
        print(format_doc_label(doc, i))
        print(doc.page_content[:preview_chars])


print("✅ Evidence display function ready.")

✅ Evidence display function ready.


In [39]:
# ==============================
# Module 4.9: Sanity tests
# 模块 4.9：快速测试
# ==============================

test_questions = [
    "What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?",
    "How does the EU AI Act define AI system?",
    "Is real-time remote biometric identification completely banned?",
    "What are the obligations for providers of general-purpose AI models with systemic risk?",
    "What does Annex III say about high-risk AI systems?"
]

for question in test_questions:
    print("\n\n" + "#" * 120)

    result = retrieve_with_evidence_reflection(
        question=question,
        max_reflection_rounds=1,
        final_top_n=10
    )

    show_evidence_result(
        result=result,
        max_docs=4,
        preview_chars=700
    )



########################################################################################################################
Question / 问题:
What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?

Question type / 问题类型:
article_specific

Supplementary retrieval rounds used / 使用的补充检索轮数:
0

Final evidence grade / 最终证据评分:
{'is_sufficient': True, 'evidence_score': 0.8821, 'score_parts': {'keyword_coverage': 0.8571428571428571, 'article_match': 1.0, 'annex_match': 0.5, 'type_specific': 1.0, 'exact_value_signal': 1.0}, 'missing_signals': [], 'question_signals': {'question_type': 'article_specific', 'mentioned_articles': ['5'], 'mentioned_annexes': [], 'requires_definition': False, 'requires_penalty': True, 'requires_gpai': False, 'requires_high_risk': False, 'requires_prohibited_practices': True, 'requires_yes_no_exceptions': False, 'requires_scenario_reasoning': False, 'requires_exact_value': True, 'extra_queries': ['Article 99 

In [40]:
# ==============================
# Module 5.1: Imports and configuration
# 模块 5.1：导入库与基础配置
# ==============================

import re
from typing import List, Dict, Any, Tuple, Optional

from langchain_core.documents import Document


# ------------------------------
# Answer generation configuration
# 答案生成配置
# ------------------------------

# Maximum number of evidence sentences used in one answer.
# 每个答案最多使用多少条证据句。
MAX_EVIDENCE_SENTENCES = 5

# Maximum answer length in characters.
# 最终答案最大字符数。
MAX_ANSWER_CHARS = 1800

# Minimum sentence length to be considered useful.
# 有效证据句的最小长度。
MIN_SENTENCE_LENGTH = 25


print("✅ Module 5 configuration loaded.")

✅ Module 5 configuration loaded.


In [41]:
# ==============================
# Module 5.2: Text and sentence utilities
# 模块 5.2：文本与句子处理工具函数
# ==============================

def clean_answer_text(text: str) -> str:
    """
    Clean final answer text.
    清洗最终答案文本。
    """
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" .", ".")
    text = text.replace(" ,", ",")
    text = text.strip()

    return text


def remove_page_markers(text: str) -> str:
    """
    Remove internal page markers from text.
    移除内部页码标记。
    """
    text = re.sub(r"\[PAGE\s+\d+\]", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_sentences(text: str) -> List[str]:
    """
    Split legal text into sentence-like units.
    将法律文本切分为近似句子单位。

    Legal text often contains long sentences, so we also split on semicolons
    and numbered list boundaries when appropriate.
    法律文本句子通常较长，因此也会适当按分号和编号列表边界切分。
    """
    text = remove_page_markers(text)

    # Normalize spaces.
    # 规范化空格。
    text = re.sub(r"\s+", " ", text).strip()

    # Split on sentence endings.
    # 按句号、问号、感叹号切分。
    parts = re.split(r"(?<=[.!?])\s+", text)

    sentences = []

    for part in parts:
        part = part.strip()

        if not part:
            continue

        # Further split very long legal sentences on semicolons.
        # 对过长法律句子进一步按分号切分。
        if len(part) > 700 and ";" in part:
            subparts = [p.strip() for p in part.split(";") if p.strip()]
            sentences.extend(subparts)
        else:
            sentences.append(part)

    # Remove too short fragments.
    # 删除过短片段。
    sentences = [
        s for s in sentences
        if len(s) >= MIN_SENTENCE_LENGTH
    ]

    return sentences


def get_all_sentences_from_docs(docs: List[Document]) -> List[Dict[str, Any]]:
    """
    Extract sentence records from retrieved documents.
    从检索文档中抽取句子记录。

    Each record keeps sentence text and document metadata.
    每条记录保留句子文本和文档 metadata。
    """
    records = []

    for doc_idx, doc in enumerate(docs):
        sentences = split_into_sentences(doc.page_content)

        for sent_idx, sent in enumerate(sentences):
            records.append({
                "text": sent,
                "doc_index": doc_idx,
                "sentence_index": sent_idx,
                "metadata": doc.metadata
            })

    return records


print("✅ Text and sentence utility functions ready.")

✅ Text and sentence utility functions ready.


In [42]:
# ==============================
# Module 5.3: Evidence sentence scoring
# 模块 5.3：证据句相关性评分
# ==============================

def get_question_keywords_for_answer(question: str) -> List[str]:
    """
    Extract useful keywords for answer sentence scoring.
    抽取用于答案句评分的关键词。
    """
    try:
        keyword_text = extract_legal_keywords(question)
    except NameError:
        keyword_text = question

    tokens = simple_tokenize(keyword_text)

    stop_tokens = {
        "what", "which", "when", "where", "how", "does",
        "under", "article", "annex", "legal", "provision",
        "requirements", "obligations", "system", "systems",
        "regulation", "european", "union"
    }

    keywords = [
        token for token in tokens
        if len(token) > 2 and token not in stop_tokens
    ]

    # Deduplicate while preserving order.
    # 去重并保留顺序。
    seen = set()
    deduped = []

    for keyword in keywords:
        if keyword not in seen:
            deduped.append(keyword)
            seen.add(keyword)

    return deduped


def score_sentence_for_question(
    sentence: str,
    question: str,
    question_type: str,
    metadata: Dict[str, Any]
) -> float:
    """
    Score a sentence according to its relevance to the question.
    根据句子与问题的相关性进行打分。
    """
    sent_norm = normalize_text(sentence)
    question_norm = normalize_text(question)

    score = 0.0

    # Keyword coverage score.
    # 关键词覆盖分。
    keywords = get_question_keywords_for_answer(question)

    if keywords:
        matched = sum(1 for kw in keywords if kw in sent_norm)
        score += 2.0 * matched / len(keywords)

    # Article / Annex metadata score.
    # Article / Annex 元数据匹配分。
    mentioned_articles = extract_question_articles(question)
    mentioned_annexes = extract_question_annexes(question)

    article_no = metadata.get("article_number")
    annex_no = metadata.get("annex_number")

    if mentioned_articles and article_no is not None:
        if str(article_no) in mentioned_articles:
            score += 1.2

    if mentioned_annexes and annex_no is not None:
        if str(annex_no).upper() in mentioned_annexes:
            score += 1.2

    # Type-specific signal score.
    # 问题类型相关信号分。
    type_terms = {
        "definition": ["means", "defined as", "definition", "refers to"],
        "penalty": ["administrative fines", "penalties", "eur", "turnover", "%"],
        "gpai": ["general-purpose ai model", "systemic risk", "flops", "high-impact capabilities"],
        "high_risk": ["high-risk ai system", "annex iii", "classification"],
        "prohibited_practices": ["prohibited", "prohibition", "unacceptable risk"],
        "yes_no": ["except", "unless", "provided that", "subject to", "shall not apply"],
        "scenario": ["provider", "deployer", "shall", "obligations", "requirements"]
    }

    for term in type_terms.get(question_type, []):
        if term in sent_norm:
            score += 0.6

    # Exact value boost.
    # 精确数值加分。
    if re.search(r"(eur|€|\d+\s*%|\d+\s*000|\d+\^\d+|10\^?\d+|\d+\s+days?|\d+\s+weeks?|\d+\s+months?|\d+\s+years?)", sent_norm):
        score += 0.7

    # Legal obligation boost.
    # 法律义务表达加分。
    if any(term in sent_norm for term in ["shall", "must", "required", "obligation", "subject to"]):
        score += 0.3

    return score


def select_top_evidence_sentences(
    question: str,
    docs: List[Document],
    question_type: str,
    max_sentences: int = MAX_EVIDENCE_SENTENCES
) -> List[Dict[str, Any]]:
    """
    Select the most relevant evidence sentences.
    选择最相关的证据句。
    """
    sentence_records = get_all_sentences_from_docs(docs)
    scored_records = []

    for record in sentence_records:
        score = score_sentence_for_question(
            sentence=record["text"],
            question=question,
            question_type=question_type,
            metadata=record["metadata"]
        )

        if score > 0:
            record = dict(record)
            record["score"] = score
            scored_records.append(record)

    scored_records.sort(key=lambda x: x["score"], reverse=True)

    # Deduplicate near-identical sentences.
    # 去除近似重复句子。
    selected = []
    seen_text = set()

    for record in scored_records:
        text_key = normalize_text(record["text"])[:180]

        if text_key in seen_text:
            continue

        selected.append(record)
        seen_text.add(text_key)

        if len(selected) >= max_sentences:
            break

    return selected


print("✅ Evidence sentence scoring functions ready.")

✅ Evidence sentence scoring functions ready.


In [43]:
# ==============================
# Module 5.4: Definition and exact fact extraction
# 模块 5.4：定义类与精确事实类答案抽取
# ==============================

def extract_defined_term(question: str) -> Optional[str]:
    """
    Try to extract the defined term from a definition question.
    尝试从定义类问题中抽取被定义术语。
    """
    patterns = [
        r"define\s+['\"]?([^'\"]+?)['\"]?\??$",
        r"definition of\s+['\"]?([^'\"]+?)['\"]?",
        r"what is meant by\s+['\"]?([^'\"]+?)['\"]?",
        r"what does the term\s+['\"]?([^'\"]+?)['\"]?",
        r"how does .* define\s+['\"]?([^'\"]+?)['\"]?"
    ]

    for pattern in patterns:
        match = re.search(pattern, question, flags=re.IGNORECASE)
        if match:
            term = match.group(1).strip()
            term = re.sub(r"\s+in\s+.*$", "", term, flags=re.IGNORECASE)
            return term.strip(" ?.")

    return None


def find_definition_sentence(question: str, evidence_sentences: List[Dict[str, Any]]) -> Optional[str]:
    """
    Find the best definition sentence.
    找到最合适的定义句。
    """
    term = extract_defined_term(question)

    if term:
        term_norm = normalize_text(term)
    else:
        term_norm = ""

    definition_patterns = [
        r"\bmeans\b",
        r"\bdefined as\b",
        r"\brefers to\b",
        r"\bshall mean\b"
    ]

    candidates = []

    for record in evidence_sentences:
        sent = record["text"]
        sent_norm = normalize_text(sent)

        has_definition_pattern = any(re.search(p, sent_norm) for p in definition_patterns)

        if has_definition_pattern:
            score = record.get("score", 0)

            if term_norm and term_norm in sent_norm:
                score += 2.0

            candidates.append((score, sent))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


def find_exact_value_sentences(evidence_sentences: List[Dict[str, Any]]) -> List[str]:
    """
    Find sentences containing exact values.
    找出包含精确数值的句子。
    """
    value_pattern = re.compile(
        r"(EUR\s*\d[\d\s,\.]*|€\s*\d[\d\s,\.]*|\d+\s*%|10\^?\d+|\d+\s+days?|\d+\s+weeks?|\d+\s+months?|\d+\s+years?)",
        flags=re.IGNORECASE
    )

    value_sentences = []

    for record in evidence_sentences:
        sent = record["text"]

        if value_pattern.search(sent):
            value_sentences.append(sent)

    return value_sentences


def add_article_prefix_if_available(answer: str, evidence_sentences: List[Dict[str, Any]]) -> str:
    """
    Add Article reference if the best evidence has metadata.
    如果最佳证据句有 Article metadata，则在答案中加入 Article 引用。
    """
    if not evidence_sentences:
        return answer

    meta = evidence_sentences[0].get("metadata", {})

    article_no = meta.get("article_number")
    annex_no = meta.get("annex_number")

    if article_no and f"Article {article_no}" not in answer:
        return f"Under Article {article_no}, {answer[0].lower() + answer[1:] if answer else answer}"

    if annex_no and f"Annex {annex_no}" not in answer:
        return f"Under Annex {annex_no}, {answer[0].lower() + answer[1:] if answer else answer}"

    return answer


print("✅ Definition and exact fact extraction functions ready.")

✅ Definition and exact fact extraction functions ready.


In [44]:
# ==============================
# Module 5.5: Question-type-aware answer builders
# 模块 5.5：按问题类型组织答案
# ==============================

def build_definition_answer(
    question: str,
    evidence_sentences: List[Dict[str, Any]]
) -> str:
    """
    Build answer for definition questions.
    为定义类问题生成答案。
    """
    definition_sentence = find_definition_sentence(question, evidence_sentences)

    if definition_sentence:
        return clean_answer_text(definition_sentence)

    return build_general_extractive_answer(question, evidence_sentences)


def build_penalty_answer(
    question: str,
    evidence_sentences: List[Dict[str, Any]]
) -> str:
    """
    Build answer for penalty / fine questions.
    为罚款 / 处罚类问题生成答案。
    """
    exact_sentences = find_exact_value_sentences(evidence_sentences)

    if exact_sentences:
        answer = " ".join(exact_sentences[:2])
        return clean_answer_text(answer)

    return build_general_extractive_answer(question, evidence_sentences)


def build_yes_no_answer(
    question: str,
    evidence_sentences: List[Dict[str, Any]]
) -> str:
    """
    Build answer for yes/no questions.
    为 Yes/No 类问题生成答案。

    The function gives a cautious legal answer based on evidence.
    该函数基于证据给出谨慎的法律回答。
    """
    question_norm = normalize_text(question)
    evidence_text = normalize_text(" ".join([r["text"] for r in evidence_sentences]))

    # Heuristic polarity detection.
    # 启发式判断答案倾向。
    has_exception = any(term in evidence_text for term in [
        "except",
        "unless",
        "provided that",
        "subject to",
        "shall not apply",
        "conditions"
    ])

    has_prohibition = any(term in evidence_text for term in [
        "prohibited",
        "shall be prohibited",
        "not allowed",
        "forbidden"
    ])

    has_obligation = any(term in evidence_text for term in [
        "shall",
        "required",
        "obligation",
        "must"
    ])

    base = build_general_extractive_answer(question, evidence_sentences)

    if "completely" in question_norm and has_exception:
        return clean_answer_text("No. The retrieved provisions indicate that the rule is subject to exceptions or conditions. " + base)

    if has_prohibition and has_exception:
        return clean_answer_text("Not completely. The retrieved provisions indicate a prohibition, but also specify exceptions or conditions. " + base)

    if has_prohibition:
        return clean_answer_text("Yes, to the extent described in the retrieved provisions. " + base)

    if has_obligation:
        return clean_answer_text("Yes, where the conditions in the Act are met. " + base)

    return clean_answer_text(base)


def build_summary_answer(
    question: str,
    evidence_sentences: List[Dict[str, Any]],
    max_points: int = 5
) -> str:
    """
    Build answer for broad summary questions.
    为较宽泛的总结类问题生成答案。
    """
    if not evidence_sentences:
        return "The retrieved evidence is insufficient to answer the question."

    points = []

    for record in evidence_sentences[:max_points]:
        sentence = clean_answer_text(record["text"])
        points.append(sentence)

    # Use compact paragraph style for Kaggle response.
    # 为 Kaggle response 使用紧凑段落形式。
    answer = " ".join(points)

    return clean_answer_text(answer)


def build_general_extractive_answer(
    question: str,
    evidence_sentences: List[Dict[str, Any]]
) -> str:
    """
    Build a general extractive answer from top evidence sentences.
    基于最相关证据句生成通用抽取式答案。
    """
    if not evidence_sentences:
        return "The retrieved evidence is insufficient to answer the question."

    selected_texts = []

    for record in evidence_sentences:
        sentence = clean_answer_text(record["text"])

        if sentence and sentence not in selected_texts:
            selected_texts.append(sentence)

        if len(selected_texts) >= MAX_EVIDENCE_SENTENCES:
            break

    answer = " ".join(selected_texts)
    answer = clean_answer_text(answer)

    if len(answer) > MAX_ANSWER_CHARS:
        answer = answer[:MAX_ANSWER_CHARS].rsplit(" ", 1)[0] + "."

    return answer


def build_answer_by_question_type(
    question: str,
    question_type: str,
    evidence_sentences: List[Dict[str, Any]]
) -> str:
    """
    Build final answer according to question type.
    根据问题类型生成最终答案。
    """
    if question_type == "definition":
        answer = build_definition_answer(question, evidence_sentences)

    elif question_type == "penalty":
        answer = build_penalty_answer(question, evidence_sentences)

    elif question_type == "yes_no":
        answer = build_yes_no_answer(question, evidence_sentences)

    elif question_type in ["scenario", "high_risk", "gpai", "prohibited_practices", "annex_specific"]:
        answer = build_summary_answer(question, evidence_sentences, max_points=5)

    else:
        answer = build_general_extractive_answer(question, evidence_sentences)

    answer = clean_answer_text(answer)

    if len(answer) > MAX_ANSWER_CHARS:
        answer = answer[:MAX_ANSWER_CHARS].rsplit(" ", 1)[0] + "."

    return answer


print("✅ Question-type-aware answer builders ready.")

✅ Question-type-aware answer builders ready.


In [45]:
# ==============================
# Module 5.6: Main local answer generation function
# 模块 5.6：本地答案生成主函数
# ==============================

def generate_local_answer_from_evidence(
    evidence_result: Dict[str, Any],
    max_sentences: int = MAX_EVIDENCE_SENTENCES
) -> Dict[str, Any]:
    """
    Generate final answer from evidence retrieval result.
    根据证据检索结果生成最终答案。

    Args:
        evidence_result:
            Output from retrieve_with_evidence_reflection()
            retrieve_with_evidence_reflection() 的输出结果。

    Returns:
        Dict containing final answer and debug information.
        返回最终答案和调试信息。
    """
    question = evidence_result["question"]
    docs = evidence_result["docs"]
    query_pack = evidence_result.get("query_pack", {})
    question_type = query_pack.get("question_type", "general")

    evidence_sentences = select_top_evidence_sentences(
        question=question,
        docs=docs,
        question_type=question_type,
        max_sentences=max_sentences
    )

    answer = build_answer_by_question_type(
        question=question,
        question_type=question_type,
        evidence_sentences=evidence_sentences
    )

    answer = add_article_prefix_if_available(
        answer=answer,
        evidence_sentences=evidence_sentences
    )

    answer = clean_answer_text(answer)

    return {
        "question": question,
        "question_type": question_type,
        "final_answer": answer,
        "evidence_sentences": evidence_sentences,
        "final_grade": evidence_result.get("final_grade", {}),
        "reflection_rounds_used": evidence_result.get("reflection_rounds_used", 0)
    }


def answer_question_locally(question: str) -> Dict[str, Any]:
    """
    End-to-end local QA function.
    端到端本地问答函数。

    Workflow:
    流程：
    question
    → retrieve_with_evidence_reflection()
    → generate_local_answer_from_evidence()
    """
    evidence_result = retrieve_with_evidence_reflection(
        question=question,
        max_reflection_rounds=MAX_REFLECTION_ROUNDS,
        final_top_n=FINAL_EVIDENCE_TOP_N
    )

    answer_result = generate_local_answer_from_evidence(
        evidence_result=evidence_result,
        max_sentences=MAX_EVIDENCE_SENTENCES
    )

    # Keep retrieval debug information.
    # 保留检索调试信息。
    answer_result["retrieval_debug"] = evidence_result.get("debug_rounds", [])

    return answer_result


print("✅ Main local answer generation functions ready.")

✅ Main local answer generation functions ready.


In [46]:
# ==============================
# Module 5.7: Display answer result
# 模块 5.7：展示答案结果
# ==============================

def show_answer_result(answer_result: Dict[str, Any], max_evidence: int = 3) -> None:
    """
    Display generated answer and supporting evidence.
    展示生成答案和支持证据。
    """
    print("=" * 120)

    print("Question / 问题:")
    print(answer_result["question"])

    print("\nQuestion type / 问题类型:")
    print(answer_result["question_type"])

    print("\nFinal answer / 最终答案:")
    print(answer_result["final_answer"])

    print("\nEvidence grade / 证据评分:")
    print(answer_result.get("final_grade", {}))

    print("\nTop evidence sentences / 主要证据句:")
    for i, record in enumerate(answer_result.get("evidence_sentences", [])[:max_evidence], start=1):
        print("\n" + "-" * 90)
        print(f"Evidence {i} | score={round(record.get('score', 0), 4)}")
        print("Metadata:", record.get("metadata", {}))
        print(record.get("text", ""))


print("✅ Answer display function ready.")

✅ Answer display function ready.


In [47]:
# ==============================
# Module 5.8: Sanity tests
# 模块 5.8：快速测试
# ==============================

test_questions = [
    "What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?",
    "How does the EU AI Act define AI system?",
    "Is real-time remote biometric identification completely banned?",
    "What are the obligations for providers of general-purpose AI models with systemic risk?",
    "What does Annex III say about high-risk AI systems?"
]

for question in test_questions:
    print("\n\n" + "#" * 120)

    answer_result = answer_question_locally(question)

    show_answer_result(
        answer_result=answer_result,
        max_evidence=3
    )



########################################################################################################################
Question / 问题:
What is the maximum administrative fine for non-compliance with the prohibition of AI practices referred to in Article 5?

Question type / 问题类型:
article_specific

Final answer / 最终答案:
Under Article 100, non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 1 500 000. Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to 35 000 000 EUR or, if the offender is an undertaking, up to 7 % of its total worldwide annual turnover for the preceding financial year, whichever is higher. The three-month period referred to in this paragraph shall be reduced to 30 days in the event of non-compliance with the prohibition of the AI practices referred to in Article 5 of this Regulation. Where, within three months

In [48]:
# ==============================
# Module 6.1: Imports and configuration
# 模块 6.1：导入库与基础配置
# ==============================

import os
import json
import time
import pandas as pd
from tqdm.notebook import tqdm


# ------------------------------
# Output files
# 输出文件
# ------------------------------

SUBMISSION_PATH = "submission.csv"
CHECKPOINT_PATH = "qa_checkpoint.jsonl"
DEBUG_PATH = "qa_debug_log.jsonl"


# ------------------------------
# Runtime configuration
# 运行配置
# ------------------------------

# Sleep time between questions.
# 每道题之间的休眠时间。纯本地运行可以设为 0。
SLEEP_SECONDS = 0

# Whether to resume from checkpoint.
# 是否从 checkpoint 断点续跑。
RESUME_FROM_CHECKPOINT = True

# Whether to save detailed debug logs.
# 是否保存详细调试日志。
SAVE_DEBUG_LOG = True


print("✅ Module 6 configuration loaded.")

✅ Module 6 configuration loaded.


In [49]:
# ==============================
# Module 6.2: Locate test CSV
# 模块 6.2：自动定位测试集 CSV
# ==============================

def find_test_csv(root_dir: str = "/kaggle/input") -> str:
    """
    Find the test CSV file under Kaggle input directory.
    在 Kaggle input 目录下自动查找测试集 CSV 文件。

    Returns:
        str: path to test CSV
        返回测试集 CSV 路径
    """
    csv_candidates = []

    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                path = os.path.join(root, file)
                csv_candidates.append(path)

    if not csv_candidates:
        raise FileNotFoundError("No CSV file found under /kaggle/input.")

    # Prefer file names related to test data.
    # 优先选择文件名中包含 test 的 CSV。
    for path in csv_candidates:
        filename = os.path.basename(path).lower()

        if "test" in filename:
            return path

    # Fallback: return the first CSV.
    # 兜底：返回第一个 CSV 文件。
    return csv_candidates[0]


test_csv_path = find_test_csv()

print(f"✅ Test CSV found: {test_csv_path}")

✅ Test CSV found: /kaggle/input/competitions/ra-gnarok/Test Data CSV - Sheet1.csv


In [50]:
# ==============================
# Module 6.3: Load test data and detect columns
# 模块 6.3：读取测试集并识别列名
# ==============================

test_df = pd.read_csv(test_csv_path)

print("✅ Test data loaded.")
print("Columns:", list(test_df.columns))
print("Number of rows:", len(test_df))
display(test_df.head())


def detect_id_and_question_columns(df: pd.DataFrame):
    """
    Detect q_id column and question column.
    自动识别 q_id 列和 question 列。

    Returns:
        tuple: (id_col, question_col)
        返回：（ID 列名，问题列名）
    """
    columns = list(df.columns)
    lower_map = {col.lower().strip(): col for col in columns}

    # Detect ID column.
    # 识别 ID 列。
    if "q_id" in lower_map:
        id_col = lower_map["q_id"]
    elif "question_id" in lower_map:
        id_col = lower_map["question_id"]
    elif "id" in lower_map:
        id_col = lower_map["id"]
    else:
        id_col = columns[0]

    # Detect question column.
    # 识别问题列。
    question_col = None

    for key in ["question", "prompt", "text", "content"]:
        if key in lower_map:
            question_col = lower_map[key]
            break

    if question_col is None:
        # Usually the question column is the second column.
        # 通常问题列是第二列。
        if len(columns) >= 2:
            question_col = columns[1]
        else:
            raise ValueError("Cannot detect question column.")

    return id_col, question_col


id_col, question_col = detect_id_and_question_columns(test_df)

print(f"✅ ID column detected: {id_col}")
print(f"✅ Question column detected: {question_col}")

✅ Test data loaded.
Columns: ['q_id', 'question']
Number of rows: 100


,q_id,question
0,Q1,What is the maximum administrative fine for no...
1,Q2,On what date did the European Parliament adopt...
2,Q3,What is the FLOP threshold above which a gener...
3,Q4,Within how many weeks must a provider notify t...
4,Q5,What does the acronym 'FLOP' stand for in the ...


✅ ID column detected: q_id
✅ Question column detected: question


In [51]:
# ==============================
# Module 6.4: Checkpoint utilities
# 模块 6.4：断点续跑工具函数
# ==============================

def load_checkpoint(path: str) -> Dict[str, Dict]:
    """
    Load completed QA records from checkpoint JSONL.
    从 checkpoint JSONL 中读取已完成记录。

    Returns:
        Dict[str, Dict]: q_id -> record
        返回：q_id 到记录的映射。
    """
    records = {}

    if not os.path.exists(path):
        return records

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                q_id = str(record["q_id"])
                records[q_id] = record
            except Exception:
                continue

    return records


def append_jsonl(path: str, record: Dict) -> None:
    """
    Append one JSON record to JSONL file.
    向 JSONL 文件追加一条记录。
    """
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def clear_file_if_needed(path: str, resume: bool) -> None:
    """
    Clear file if not resuming.
    如果不进行断点续跑，则清空旧文件。
    """
    if not resume and os.path.exists(path):
        os.remove(path)


# Clear old files if needed.
# 如果不续跑，则清空旧 checkpoint 和 debug。
clear_file_if_needed(CHECKPOINT_PATH, RESUME_FROM_CHECKPOINT)
clear_file_if_needed(DEBUG_PATH, RESUME_FROM_CHECKPOINT)


completed_records = load_checkpoint(CHECKPOINT_PATH) if RESUME_FROM_CHECKPOINT else {}

print(f"✅ Loaded completed records: {len(completed_records)}")

✅ Loaded completed records: 0


In [52]:
# ==============================
# Module 6.5: Single question processing
# 模块 6.5：单题处理函数
# ==============================

def process_single_question(q_id: str, question: str) -> Dict:
    """
    Process one question end-to-end.
    端到端处理一道问题。

    Workflow:
    流程：
    question
    → answer_question_locally()
    → final response
    → debug record
    """
    start_time = time.time()

    try:
        answer_result = answer_question_locally(question)

        final_answer = str(answer_result.get("final_answer", "")).strip()

        if not final_answer:
            final_answer = "The retrieved evidence is insufficient to answer the question."

        status = "success"
        error_message = None

    except Exception as e:
        answer_result = {}
        final_answer = "Error generating response."
        status = "error"
        error_message = str(e)

    elapsed_time = time.time() - start_time

    record = {
        "q_id": str(q_id),
        "question": str(question),
        "response": final_answer,
        "status": status,
        "error": error_message,
        "elapsed_time": round(elapsed_time, 3),

        # Useful debug fields.
        # 有用的调试字段。
        "question_type": answer_result.get("question_type"),
        "final_grade": answer_result.get("final_grade"),
        "reflection_rounds_used": answer_result.get("reflection_rounds_used"),
        "evidence_sentences": [
            {
                "score": item.get("score"),
                "text": item.get("text"),
                "metadata": item.get("metadata")
            }
            for item in answer_result.get("evidence_sentences", [])
        ],
        "retrieval_debug": answer_result.get("retrieval_debug", [])
    }

    return record


print("✅ Single question processor ready.")

✅ Single question processor ready.


In [53]:
# ==============================
# Module 6.6: Batch processing
# 模块 6.6：批量处理所有问题
# ==============================

results_map = dict(completed_records)

print(f"🚀 Starting batch QA.")
print(f"Total questions: {len(test_df)}")
print(f"Already completed: {len(results_map)}")

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating answers"):
    q_id = str(row[id_col])
    question = str(row[question_col])

    # Skip completed questions.
    # 跳过已经完成的问题。
    if q_id in results_map:
        continue

    record = process_single_question(
        q_id=q_id,
        question=question
    )

    # Save checkpoint immediately.
    # 立即保存 checkpoint，避免中断后丢失进度。
    append_jsonl(CHECKPOINT_PATH, record)

    # Save debug log if enabled.
    # 如果开启，则保存 debug 日志。
    if SAVE_DEBUG_LOG:
        append_jsonl(DEBUG_PATH, record)

    results_map[q_id] = record

    if SLEEP_SECONDS > 0:
        time.sleep(SLEEP_SECONDS)


print("✅ Batch processing finished.")
print(f"Completed records: {len(results_map)}")

🚀 Starting batch QA.
Total questions: 100
Already completed: 0


Generating answers:   0%|          | 0/100 [00:00<?, ?it/s]

✅ Batch processing finished.
Completed records: 100


In [54]:
# ==============================
# Module 6.7: Build submission.csv
# 模块 6.7：生成 submission.csv
# ==============================

submission_rows = []

for _, row in test_df.iterrows():
    q_id = str(row[id_col])

    if q_id in results_map:
        response = results_map[q_id].get("response", "")
    else:
        response = "Error generating response."

    submission_rows.append({
        "q_id": q_id,
        "response": str(response).strip()
    })


submission_df = pd.DataFrame(submission_rows)

# Ensure exact output columns.
# 确保输出列名严格为 q_id,response。
submission_df = submission_df[["q_id", "response"]]

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"✅ Submission saved to: {SUBMISSION_PATH}")
print("Submission columns:", list(submission_df.columns))
print("Number of rows:", len(submission_df))

display(submission_df.head())

✅ Submission saved to: submission.csv
Submission columns: ['q_id', 'response']
Number of rows: 100


,q_id,response
0,Q1,"Under Article 99, non-compliance with the proh..."
1,Q2,"Under Article 99, non-compliance with the proh..."
2,Q3,"Under Article 51, a general-purpose AI model s..."
3,Q4,A general-purpose AI model shall be presumed t...
4,Q5,"Under Article 51, a general-purpose AI model s..."


In [55]:
# ==============================
# Module 6.8: Submission quality checks
# 模块 6.8：提交文件质量检查
# ==============================

def check_submission(submission_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    """
    Run basic quality checks for submission.
    对 submission 进行基础质量检查。
    """
    print("========== Submission Check ==========")

    # Check columns.
    # 检查列名。
    expected_cols = ["q_id", "response"]
    print("Columns correct:", list(submission_df.columns) == expected_cols)

    # Check row count.
    # 检查行数。
    print("Row count matches test:", len(submission_df) == len(test_df))

    # Check missing responses.
    # 检查空答案。
    missing_count = submission_df["response"].isna().sum()
    empty_count = (submission_df["response"].astype(str).str.strip() == "").sum()

    print("Missing responses:", missing_count)
    print("Empty responses:", empty_count)

    # Check duplicate q_id.
    # 检查重复 q_id。
    duplicate_count = submission_df["q_id"].duplicated().sum()
    print("Duplicate q_id:", duplicate_count)

    # Basic answer length statistics.
    # 答案长度统计。
    lengths = submission_df["response"].astype(str).str.len()
    print("\nAnswer length statistics:")
    print(lengths.describe())

    print("\nShortest answers:")
    display(submission_df.assign(length=lengths).sort_values("length").head(5))

    print("\nLongest answers:")
    display(submission_df.assign(length=lengths).sort_values("length", ascending=False).head(5))


check_submission(submission_df, test_df)

========== Submission Check ==========
Columns correct: True
Row count matches test: True
Missing responses: 0
Empty responses: 0
Duplicate q_id: 0

Answer length statistics:
count     100.00000
mean     1461.89000
std       303.58717
min       452.00000
25%      1257.25000
50%      1475.50000
75%      1784.75000
max      1818.00000
Name: response, dtype: float64

Shortest answers:


,q_id,response,length
83,Q84,"Under Article 3, (23) ‘substantial modificatio...",452
1,Q2,"Under Article 99, non-compliance with the proh...",640
67,Q68,"Under Article 99, non-compliance with the proh...",640
88,Q89,In addition to the high-risk AI systems referr...,843
18,Q19,"Under Article 3, (4) ‘deployer’ means a natura...",867



Longest answers:


,q_id,response,length
79,Q80,"Under Article 74, without prejudice to the pow...",1818
58,Q59,"Under Article 57, the European Data Protection...",1817
95,Q96,"Under Article 51, a general-purpose AI model s...",1817
5,Q6,"Under Article 100, before taking decisions pur...",1815
68,Q69,"Under Article 5, (g) the placing on the market...",1814


In [56]:
# ==============================
# Module 6.9: Inspect low-confidence or failed cases
# 模块 6.9：查看低置信度或失败样本
# ==============================

def load_jsonl_records(path: str) -> List[Dict]:
    """
    Load records from JSONL file.
    从 JSONL 文件读取记录。
    """
    records = []

    if not os.path.exists(path):
        return records

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except Exception:
                continue

    return records


debug_records = load_jsonl_records(CHECKPOINT_PATH)

inspection_rows = []

for record in debug_records:
    grade = record.get("final_grade") or {}
    evidence_score = grade.get("evidence_score", None)

    inspection_rows.append({
        "q_id": record.get("q_id"),
        "status": record.get("status"),
        "question_type": record.get("question_type"),
        "evidence_score": evidence_score,
        "reflection_rounds_used": record.get("reflection_rounds_used"),
        "question": record.get("question"),
        "response": record.get("response")
    })


inspection_df = pd.DataFrame(inspection_rows)

if len(inspection_df) > 0:
    print("Low evidence score cases:")
    display(
        inspection_df
        .sort_values(["status", "evidence_score"], ascending=[True, True])
        .head(10)
    )
else:
    print("No debug records found.")

Low evidence score cases:


,q_id,status,question_type,evidence_score,reflection_rounds_used,question,response
80,Q81,success,general,0.5000,1,In plain language: what does the Act say about...,(104) The providers of general-purpose AI mode...
31,Q32,success,general,0.5300,1,How does the EU AI Act treat open-source AI sy...,(103) Free and open-source AI components cover...
63,Q64,success,general,0.5375,1,What does the EU AI Act say about AI regulator...,"Under Article 58, the implementing acts referr..."
62,Q63,success,general,0.5500,0,Describe the entire complaint and remedy ecosy...,Once a harmonised standard is published and as...
75,Q76,success,general,0.5500,0,In simpler terms: what is the EU AI Act trying...,(1) The purpose of this Regulation is to impro...
79,Q80,success,general,0.5600,0,"Without using technical jargon, explain what '...","Under Article 74, without prejudice to the pow..."
82,Q83,success,general,0.5643,0,"In your own words, what does 'CE marking' mean...","Under Article 41, before preparing a draft imp..."
76,Q77,success,general,0.5714,0,"In your own words, what does the Act mean when...","Under Article 111, without prejudice to the ap..."
77,Q78,success,general,0.6000,0,What is the EU AI Act essentially saying when ...,Any known or foreseeable circumstances related...
78,Q79,success,general,0.6000,0,Rephrase: what does the Act mean by a 'post-ma...,"Under Article 12, in order to ensure a level o..."


In [57]:
# ==============================
# Module 6.10: Final file check
# 模块 6.10：最终文件检查
# ==============================

print("Generated files / 已生成文件:")

for path in [SUBMISSION_PATH, CHECKPOINT_PATH, DEBUG_PATH]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"✅ {path} | {size_kb:.2f} KB")
    else:
        print(f"⚠️ {path} not found")

Generated files / 已生成文件:
✅ submission.csv | 143.91 KB
✅ qa_checkpoint.jsonl | 1132.10 KB
✅ qa_debug_log.jsonl | 1132.10 KB


In [58]:
!zip -r project_outputs.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/chunks_legal_structure.jsonl (deflated 79%)
  adding: kaggle/working/qa_debug_log.jsonl (deflated 88%)
  adding: kaggle/working/submission.csv (deflated 81%)
  adding: kaggle/working/qa_checkpoint.jsonl (deflated 88%)
  adding: kaggle/working/faiss_eu_ai_act_index/ (stored 0%)
  adding: kaggle/working/faiss_eu_ai_act_index/index.faiss (deflated 7%)
  adding: kaggle/working/faiss_eu_ai_act_index/index.pkl (deflated 74%)
  adding: kaggle/working/__notebook__.ipynb (deflated 80%)
